## Helper functions and imports

Libraries

In [ ]:
# functions

from ensemble_chaos_tools import check_chaos
from earthkit.regrid import interpolate

import xarray as xr
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.ticker as mtickers
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
from matplotlib import rcParams
import cartopy.crs as ccrs
import nicopal as ncp
import json

import metpy.calc as mpcalc
from metpy.interpolate import interpolate_to_grid
from metpy.units import units
from pyextremes import EVA
from colindex2 import Detect
from scipy import stats

from tqdm.notebook import tqdm
import os
import glob
import pickle
import math
import shutil

%load_ext autoreload
%autoreload 2
#%config InlineBackend.figure_format="jpeg"

In [ ]:
def setup_latex_style(base_size=10):
    rcParams.update(
        {
            "font.size": base_size,
            "axes.titlesize": base_size,
            "axes.labelsize": base_size -1,
            "xtick.labelsize": base_size - 1,
            "ytick.labelsize": base_size - 1,
            "legend.fontsize": base_size - 1,
            "font.family": "serif",
            "font.serif": ["cmr10"],
            "mathtext.fontset": "cm",
            "axes.formatter.use_mathtext": True,
            "contour.linewidth": 0.6,
        }
    )


def get_figsize(width_pt, fraction=1.0, height_factor=0.75):
    inches_per_pt = 1.0 / 72.27
    fig_width_in = width_pt * fraction * inches_per_pt
    fig_height_in = fig_width_in * height_factor
    return (fig_width_in, fig_height_in)


setup_latex_style()

WIDTH_PAPER = 372.0
WIDTH_INSA = 496.0

N320 grid coordinates

In [ ]:
with open("/homedata/pchevali/n320_coordinates.pkl", "rb") as f:
    grid = pickle.load(f)

Some useful functions

In [ ]:
def get_nearest_point(ds, target_lat, target_lon):
    distances = (ds["latitude"] - target_lat) ** 2 + (ds["longitude"] - target_lon) ** 2
    return distances.argmin().compute().item()

Figure out the mask for outputs of aifs

In [ ]:
mask = (
    (grid["latitude"] > 15)
    * (grid["latitude"] < 75)
    * ((grid["longitude"] > 300) + (grid["longitude"] < 30))
)
with open("/homedata/pchevali/mask_eu.pkl", "wb") as f:
    pickle.dump(mask, f)

Import AIFS Simulations

In [ ]:
dates = [
    "2026051500",
    "2026051600",
    "2026051700",
    "2026051800",
    "2026051900",
    "2026052000",
    "2026052100",
    "2026052200",
    "2026052300",
]
variables = [
    "2d",
    "2t",
    "sp",
    "tp",
    "t_200",
    "t_250",
    "t_300",
    "t_400",
    "t_500",
    "t_600",
    "t_700",
    "t_850",
    "t_925",
    "t_1000",
    "u_200",
    "u_250",
    "u_300",
    "u_400",
    "u_500",
    "u_600",
    "u_700",
    "u_850",
    "u_925",
    "u_1000",
    "v_200",
    "v_250",
    "v_300",
    "v_400",
    "v_500",
    "v_600",
    "v_700",
    "v_850",
    "v_925",
    "v_1000",
    "w_200",
    "w_250",
    "w_300",
    "w_400",
    "w_500",
    "w_600",
    "w_700",
    "w_850",
    "w_925",
    "w_1000",
    "z_200",
    "z_250",
    "z_300",
    "z_400",
    "z_500",
    "z_600",
    "z_700",
    "z_850",
    "z_925",
    "z_1000",
]

base_dir = "/scratchx/pchevali/MAY_2026_HEATWAVE_PROCESSED"

datasets_results = {}

for date in dates:
    datasets_results[date] = {var: {} for var in variables}

    for var in variables:
        pattern = f"{base_dir}/{date}-*/*-{var}.nc"
        file_paths = glob.glob(pattern)

        ds = xr.open_dataset(
            file_paths[0], decode_timedelta=True, chunks="auto", engine="h5netcdf"
        )
        ds = ds.assign_coords(longitude=(((ds.longitude + 180) % 360) - 180))
        datasets_results[date][var] = ds[var]

for date in dates:
    datasets_results[date] = xr.merge(
        datasets_results[date].values(), compat="override"
    )

Indexing the latitude and longitude from the AIFS runs (and some common points and zones)

In [ ]:
LON = datasets_results["2026051500"].longitude.values
LAT = datasets_results["2026051500"].latitude.values

In [ ]:
paris_latitude = 49
paris_longitude = 2.5

plot_region_latitude_bnd = slice(23, 67)
plot_region_longitude_bnd = slice(-30, 40)

average_latitude_bnd = slice(44.5, 55)
average_longitude_bnd = slice(-5, 5.5)

cutoff_region_lat_bnd = slice(-13, -7)
cutoff_region_lon_bnd = slice(41, 43)

anticyclone_center_region_lat_bnd = slice(48, 50)
anticyclone_center_region_lon_bnd = slice(0, 6)

average_region = (
    (LAT >= average_latitude_bnd.start)
    & (LAT <= average_latitude_bnd.stop)
    & (LON >= average_longitude_bnd.start)
    & (LON <= average_longitude_bnd.stop)
)

plot_region_indexs = (
    (LAT >= plot_region_latitude_bnd.start)
    & (LAT <= plot_region_latitude_bnd.stop)
    & (LON >= plot_region_longitude_bnd.start)
    & (LON <= plot_region_longitude_bnd.stop)
)

cutoff_region_indexs = (
    (LAT >= cutoff_region_lat_bnd.start)
    & (LAT <= cutoff_region_lat_bnd.stop)
    & (LON >= cutoff_region_lon_bnd.start)
    & (LON <= cutoff_region_lon_bnd.stop)
)

anticyclone_center_region_indexs = (
    (LAT >= anticyclone_center_region_lat_bnd.start)
    & (LAT <= anticyclone_center_region_lat_bnd.stop)
    & (LON >= anticyclone_center_region_lon_bnd.start)
    & (LON <= anticyclone_center_region_lon_bnd.stop)
)

anticyclone_search_lat_bnd = slice(35, 60)
anticyclone_search_lon_bnd = slice(-25, 20)

anticyclone_search_region_indexs = (
    (LAT >= anticyclone_search_lat_bnd.start)
    & (LAT <= anticyclone_search_lat_bnd.stop)
    & (LON >= anticyclone_search_lon_bnd.start)
    & (LON <= anticyclone_search_lon_bnd.stop)
)

paris_big_box_lon = slice(0, 5)
paris_big_box_lat = slice(46, 50)
paris_big_box_region = (
    (LAT >= paris_big_box_lat.start)
    & (LAT <= paris_big_box_lat.stop)
    & (LON >= paris_big_box_lon.start)
    & (LON <= paris_big_box_lon.stop)
)

nantes_latitude = 47.22
nantes_longitude = -1.55

nantes_region_lat_bnd = slice(46, 48)
nantes_region_lon_bnd = slice(-2.1, -0.5)

nantes_region_indexs = (
    (LAT >= nantes_region_lat_bnd.start)
    & (LAT <= nantes_region_lat_bnd.stop)
    & (LON >= nantes_region_lon_bnd.start)
    & (LON <= nantes_region_lon_bnd.stop)
)

nantes_index = get_nearest_point(
    datasets_results["2026051500"], nantes_latitude, nantes_longitude
)

paris_index = get_nearest_point(
    datasets_results["2026051500"], paris_latitude, paris_longitude
)

ERA5 climatology for may

In [ ]:
climato_era5 = xr.open_dataset(
    "/homedata/pchevali/ERA5_CLIMATO/t2m_may_era5_climato.nc"
)
climato_era5 = climato_era5.isel(hour=climato_era5.hour.isin([12]))
climato_era5 = interpolate(
    climato_era5["t2m"].values, {"grid": [0.25, 0.25]}, {"grid": "N320"}
)
climato_era5 = xr.DataArray(
    data=climato_era5,
    dims=["values"],
    coords={
        "latitude": ("values", grid["latitude"]),
        "longitude": ("values", grid["longitude"]),
    },
)
climato_era5 = climato_era5.assign_coords(
    longitude=(((climato_era5.longitude + 180) % 360) - 180)
).values[mask]

In [ ]:
climato_era5_ta = xr.open_dataset(
    "/homedata/pchevali/ERA5_CLIMATO/ta_may_era5_climato.nc"
)

output_datasets = []

for hour in climato_era5_ta.hour.values:
    level_datasets = [] 
    for level in climato_era5_ta.level.values:
        interpolated_values = interpolate(
            climato_era5_ta.sel(hour=hour, level=level)["ta"].values, 
            in_grid={"grid": [0.25, 0.25]}, 
            out_grid={"grid": "N320"}
        )
        interpolated_da = xr.DataArray(
            interpolated_values,
            dims=["values"],
            coords={
                "latitude": ("values", grid["latitude"]),
                "longitude": ("values", grid["longitude"]),
                "hour": hour,
                "level": level
            }
        )
        level_datasets.append(interpolated_da) 
    output_datasets.append(level_datasets) 

climato_era5_ta_interpolated = xr.combine_nested(
    output_datasets, 
    concat_dim=["hour", "level"]
)-0.75

climato_era5_ta_interpolated = climato_era5_ta_interpolated.assign_coords(
    longitude=(((climato_era5_ta_interpolated.longitude + 180) % 360) - 180)
)

climato_era5_ta_interpolated = climato_era5_ta_interpolated.isel(values=mask)

In [ ]:
era5_paris_ts=xr.open_dataset("reanalysis-era5-single-levels-timeseries-sfcq047uaxj.nc")
max_paris_era5_00utc=era5_paris_ts.t2m.sel(valid_time=era5_paris_ts.valid_time.dt.hour.isin([0])).max().item()-273.15
max_paris_era5_12utc=era5_paris_ts.t2m.sel(valid_time=era5_paris_ts.valid_time.dt.hour.isin([12])).max().item()-273.15
max_paris_era5_18utc=era5_paris_ts.t2m.sel(valid_time=era5_paris_ts.valid_time.dt.hour.isin([18])).max().item()-273.15
print(max_paris_era5_00utc,max_paris_era5_12utc,max_paris_era5_18utc)

In [ ]:
era5_london_ts=xr.open_dataset("era5_london_ts.nc")

Wrapper around colindex2 to detect cutoffs

In [ ]:
def detect_cutoffs(
    base_dir, forecast_init_date, target_var, lat_min, lat_max, lon_min, lon_max
):
    """Detect and extract cutoff low pressure systems from forecast data.

    Uses metpy to interpolate 1D unstructured grids to a 2D mesh, then applies
    colindex2.Detect to identify low pressure areas. Filters the results for a
    specific geographic bounding box and intensity threshold (So >= 6).

    Args:
        base_dir: Root directory containing the processed forecast NetCDF files.
        forecast_init_date: String representing the forecast initialization date (YYYYMMDDHH).
        target_var: Variable name representing geopotential (e.g., 'z_500').
        lat_min: Minimum latitude for the detection bounding box.
        lat_max: Maximum latitude for the detection bounding box.
        lon_min: Minimum longitude for the detection bounding box.
        lon_max: Maximum longitude for the detection bounding box.

    Returns:
        None. Saves the filtered cutoff low detection results to a CSV file.
    """
    # load dataset
    pattern = f"{base_dir}/{forecast_init_date}-*/*-{target_var}.nc"
    file_paths = glob.glob(pattern)
    v = xr.open_dataset(file_paths[0], decode_timedelta=True, engine="h5netcdf")
    v = v.assign_coords(longitude=(((v.longitude + 180) % 360) - 180)) / 9.81
    v = v.swap_dims({"step": "valid_time"})

    target_days = [
        "2026-05-20",
        "2026-05-21",
        "2026-05-22",
        "2026-05-23",
        "2026-05-24",
        "2026-05-25",
    ]

    # n° of members
    total_members = len(v.number)

    # extract lat and lon for metpy
    lon_1d = v["longitude"].compute().values
    lat_1d = v["latitude"].compute().values

    final_results = []

    # compute loop, using metpy for regridding and colindex2 for detecting low pressure areas
    for day_str in target_days:

        v_daily_mean = v.sel(valid_time=day_str).mean(dim="valid_time")

        current_ts = pd.Timestamp(day_str)
        ym_str = current_ts.strftime("%Y%m")
        filename = f"V-L-{current_ts.strftime('%Y%m%d0000')}-0500.csv"

        print(f"\n========== Processing Target Date: {current_ts} ==========")
        for member in tqdm(range(total_members)):
            # extract data to use
            z_500_1d = v_daily_mean[target_var].isel(number=member).compute().values

            # regrid directly on the fly
            grid_lon, grid_lat, z_500_2d = interpolate_to_grid(
                lon_1d, lat_1d, z_500_1d, interp_type="linear", hres=1.25
            )

            # extract lats and lon for new grid
            lons_axis = grid_lon[0, :]
            lats_axis = grid_lat[:, 0]

            custom_odir = f"./d01_tmp/day_{day_str}_member_{member}"

            # run colindex2
            Detect(
                da=z_500_2d,
                odir=custom_odir,
                ty="L",
                lev=500,
                lons=lons_axis,
                lats=lats_axis,
                t=[current_ts],
                r=np.arange(200, 1001, 100),
                So_thres=3.0,
                SR_thres=3.0,
                Do_thres=0.0,
                xx_thres=0.0,
                rm_rmin=True,
                rm_rmax=True,
                local_ex_req_num=7,
            )

            # read and filter results
            csv_path = f"{custom_odir}/V/{ym_str}/{filename}"
            if os.path.exists(csv_path):
                df = pd.read_csv(csv_path)
                if not df.empty:
                    # Filter for stuff over iberian peninsula and So>6
                    df_peninsula = df[
                        (df["lat"] >= lat_min)
                        & (df["lat"] <= lat_max)
                        & (df["lon"] >= lon_min)
                        & (df["lon"] <= lon_max)
                        & (df["So"] >= 6)
                    ].copy()

                    if not df_peninsula.empty:
                        df_peninsula["EEang_deg"] = np.degrees(df_peninsula["EEang"])

                        # add more data about the event
                        df_peninsula["Target_Date"] = current_ts
                        df_peninsula["Forecast_Init"] = forecast_init_date
                        df_peninsula["Member"] = member

                        up_or_down_list = []
                        for _, row in df_peninsula.iterrows():
                            target_lat = lat_1d[np.abs(lat_1d - row["lat"]).argmin()]
                            valid_mask = lat_1d == target_lat

                            if valid_mask.any():
                                max_lon = lon_1d[valid_mask][
                                    np.argmax(z_500_1d[valid_mask])
                                ]

                                if max_lon > row["lon"]:
                                    up_or_down_list.append("up")
                                else:
                                    up_or_down_list.append("down")
                            else:
                                up_or_down_list.append(pd.NA)

                        df_peninsula["up_or_down"] = up_or_down_list

                        clean_df = df_peninsula[
                            [
                                "Target_Date",
                                "Forecast_Init",
                                "Member",
                                "lon",
                                "lat",
                                "So",
                                "up_or_down",
                                "Do",
                                "ro",
                                "ex",
                                "SR",
                                "m",
                                "EEang_deg",
                                "EE",
                            ]
                        ]

                        final_results.append(clean_df)

            # clean temp folder
            shutil.rmtree(custom_odir, ignore_errors=True)

    master_df = pd.concat(final_results, ignore_index=True)
    output_csv = f"cols_index_Init_{forecast_init_date}.csv"
    master_df.to_csv(output_csv, index=False)
    os.remove("_stencil_coords.npy")

Functions for the MSE

In [ ]:
z = xr.open_dataset(
    "/scratchx/pchevali/z_cutout.nc",
)

In [ ]:
def moist_static_energy(T_s, T_500, T_2d, sp, z_500):
    """Calculate the surface Moist Static Energy and the saturation MSE at 500 hPa.

    Applies Tetens' formula to compute vapor pressure and evaluates the thermodynamic
    equations for surface moist static energy and 500 hPa saturation moist static energy.

    Args:
        T_s: xarray DataArray of surface temperatures (K).
        T_500: xarray DataArray of temperatures at 500 hPa (K).
        T_2d: xarray DataArray of 2-meter dew point temperatures (K).
        sp: xarray DataArray of surface pressure (Pa).
        z_500: xarray DataArray of 500 hPa geopotential height (m^2/s^2).

    Returns:
        MSE_500_star, MSE_s: Tuple of xarray DataArrays containing the saturation 
                             moist static energy at 500 hPa and the surface moist 
                             static energy, respectively (J/kg).
    """
    # constants
    c_p = 1004.0  # Specific heat of air at constant pressure (J/kg/K)
    L_v = 2.5e6  # Latent heat of vaporization (J/kg)
    epsilon = 0.622  # Molar ratio of water vapor to dry air

    def calc_vapor_pressure(T_kelvin):
        # tetens formula (gives kpa but we want pa)
        T_celsius = T_kelvin - 273.15
        e_hpa = 6.1094 * np.exp((17.625 * T_celsius) / (T_celsius + 243.04))
        return e_hpa * 100

    e_actual = calc_vapor_pressure(T_2d)
    q_s = epsilon * (e_actual / sp)
    MSE_s = (c_p * T_s) + (L_v * q_s) + z["z"]

    e_sat_500 = calc_vapor_pressure(T_500)
    q_sat_500 = epsilon * (e_sat_500 / 50000)
    MSE_500_star = (c_p * T_500) + (L_v * q_sat_500) + z_500

    return MSE_500_star, MSE_s


def MSE_compute_and_plot(dataset, region=average_region, title=None):
    """Compute and plot the Moist Static Energy, precipitation, and temperature.

    Calculates the spatial average of MSE over a specific region and generates a 
    three-panel plot displaying MSE instability (surface vs. 500 hPa), total 
    precipitation, and surface temperature over time.

    Args:
        dataset: xarray Dataset containing variables '2t', 't_500', '2d', 'sp', 'z_500', and 'tp'.
        region: Boolean mask or slice for spatial averaging. Defaults to average_region.
        title: Title string used to name the output plot file.

    Returns:
        None. Displays a matplotlib figure and saves it as a PDF.
    """

    tp = (dataset["tp"] * 1000).squeeze()  # millimeters

    MSE_500_star, MSE_s = moist_static_energy(
        dataset["2t"], dataset["t_500"], dataset["2d"], dataset["sp"], dataset["z_500"]
    )
    x_axis = dataset["tp"].step.values / np.timedelta64(1, "D")

    mse_500_plot = (
        MSE_500_star.isel(values=region).mean("values").squeeze().compute() / 1000
    )
    mse_s_plot = MSE_s.isel(values=region).mean("values").squeeze().compute() / 1000
    tp_plot = tp.isel(values=region).mean("values").squeeze().compute()
    T_s_plot = (
        dataset["2t"].isel(values=region).mean("values").squeeze().compute() - 273.15
    )

    fig, (ax1, ax2, ax3) = plt.subplots(
        3,
        1,
        figsize=get_figsize(WIDTH_INSA, 1, 0.7),
        sharex=True,
        gridspec_kw={"height_ratios": [1.5, 1, 1]},
    )

    #### MSE plot

    ax1.plot(x_axis, mse_s_plot, color="limegreen", label="$MSE_s$ (Surface)")
    ax1.plot(
        x_axis,
        mse_500_plot,
        color="darkgreen",
        label="$MSE_{500}^*$ (500hPa Saturation)",
    )
    ax1.fill_between(
        x_axis,
        mse_s_plot,
        mse_500_plot,
        where=(mse_s_plot > mse_500_plot),
        color="red",
        interpolate=True,
        alpha=0.1,
        label="Instability",
    )  # periods where mse_s-mse_500>0

    ax1.set_ylabel("Moist Static Energy (J/g)")
    ax1.grid(ls=":")
    ax1.legend(loc="best", frameon=True)

    # precipitation
    ax2.fill_between(x_axis, tp_plot, 0, color="blue", alpha=0.3)
    ax2.plot(x_axis, tp_plot, color="blue", label="Total Precipitation")
    ax2.set_ylabel("Precipitation (mm/6h)")
    ax2.grid(ls=":")

    # temperature
    ax3.plot(x_axis, T_s_plot, color="black", label="Surface Temperature ($T_s$)")
    ax3.set_ylabel("Temperature ($^\circ$C)")
    ax3.grid(ls=":")
    ax3.set_xlabel("Time (Days)")
    ax3.set_xlim(x_axis.min(), x_axis.max())

    # fig.suptitle(title)
    # plt.tight_layout()
    plt.savefig(
        f"atmospheric_profile_{title.replace(' ', '_')}.pdf", bbox_inches="tight"
    )
    plt.show()

# Analysis

## Some plots of plumes

In [ ]:
for k, v in datasets_results.items():
    if k in ["2026051700", "2026051800", "2026051900", "2026052000", "2026052100", "2026052200", "2026052300"]:
        check_chaos(
            (v["2t"].isel(values=paris_index) - 273.15),
            f"Forecasts starting {k}",
            time_of_day=[12],
            era=era5_paris_ts["t2m"]-273.15,
            save=f"{k}_t2m_paris"
        )

mettre la plus chaude en avant

In [ ]:
for k, v in datasets_results.items():
    check_chaos(
        (v["z_500"].isel(values=average_region).mean("values")),
        f"Forecasts starting {k}",
        time_of_day=[0, 6, 12, 18],
        hline=57500,
    )

## Predictability barrier (members exceeding 57500)

### Mine

In [ ]:
date_25 = np.datetime64("2026-05-25T12:00")
date_26 = np.datetime64("2026-05-26T12:00")

labels = []
probs_25 = []
probs_26 = []
probs_both = []
members_exceeding = {}

for k, v in datasets_results.items():
    labels.append(pd.to_datetime(k, format="%Y%m%d%H%M").strftime("%b %d"))

    z500_mean = v["z_500"].isel(values=average_region).mean("values")

    idx_25 = v.valid_time.values == date_25
    idx_26 = v.valid_time.values == date_26

    mask_25 = (z500_mean.isel(step=idx_25) > 57500).values.squeeze()
    mask_26 = (z500_mean.isel(step=idx_26) > 57500).values.squeeze()

    mask_both = mask_25 & mask_26

    prob_25 = np.mean(mask_25)
    prob_26 = np.mean(mask_26)
    prob_both = np.mean(mask_both)

    probs_25.append(prob_25)
    probs_26.append(prob_26)
    probs_both.append(prob_both)
    members_exceeding[k] = v["number"].values[mask_both]

## PLOT

df = pd.DataFrame(
    {"May 25th": probs_25, "May 26th": probs_26, "Both Days": probs_both}, index=labels
)
ax = df.plot.bar(figsize=(12, 6), rot=0, color=["skyblue", "coral", "purple"])
ax.set_ylabel("Probability")
ax.set_xlabel("Forecast starting date")
ax.set_title(
    "Probability of $z_{500} > 57500$ $\\text{m}^2\\text{s}^{-2}$", fontsize="x-large"
)
ax.grid(axis="y", linestyle=":")
ax.legend(fontsize="large")
plt.tight_layout()
plt.savefig("MAY_2026_HEATWAVE/predict_barrier.pdf",bbox_inches="tight")
plt.show()

### dl era5 25th

In [ ]:
import os
import pickle
import numpy as np
import earthkit.data as ekd
import earthkit.regrid as ekr

os.environ["CDSAPI_RC"] = "/home/pchevali/.cdsapirc"

# ==========================================
# 1. Settings
# ==========================================
cds_year = "2026"  
target_month = "05"
target_day = "25"
target_date = f"{cds_year}-{target_month}-{target_day}"

times = ["00:00", "06:00", "12:00", "18:00"]

save_path = f"/homedata/pchevali/AIFS_INPUTS/era5_daily_mean_{cds_year}{target_month}{target_day}.pkl"

print(f"Fetching ERA5 data for {target_date} at {times}...")

# ==========================================
# 2. Fetch & Process Surface Temperature (2t)
# ==========================================
print("Downloading 2t (Surface)...")
ds_sfc = ekd.from_source(
    "cds",
    "reanalysis-era5-single-levels",
    product_type="reanalysis",
    param=["2t"],
    date=target_date,
    time=times,
    grid=[0.25, 0.25],
)

# Extract the 4 time steps and average them across the time axis (axis=0)
t2m_values = np.stack([f.to_numpy() for f in ds_sfc])
t2m_daily_mean = t2m_values.mean(axis=0)

print("Interpolating 2t to N320...")
t2m_n320 = ekr.interpolate(t2m_daily_mean, {"grid": (0.25, 0.25)}, {"grid": "N320"})


# ==========================================
# 3. Fetch & Process Geopotential (z500)
# ==========================================
print("Downloading z500 (Pressure Levels)...")
ds_pl = ekd.from_source(
    "cds",
    "reanalysis-era5-pressure-levels",
    product_type="reanalysis",
    param=["z"],
    pressure_level=[500],
    date=target_date,
    time=times,
    grid=[0.25, 0.25],
)

# Extract the 4 time steps and average them
z500_values = np.stack([f.to_numpy() for f in ds_pl])
z500_daily_mean = z500_values.mean(axis=0)

print("Interpolating z500 to N320...")
z500_n320 = ekr.interpolate(z500_daily_mean, {"grid": (0.25, 0.25)}, {"grid": "N320"})


# ==========================================
# 4. Save to Pickle (Matching your existing format)
# ==========================================
print(f"Saving to {save_path}...")


era5_state = {
    "date": target_date,
    "fields": {
        "2t": [None, t2m_n320], 
        "z_500": [None, z500_n320]
    }
}

with open(save_path, "wb") as f:
    pickle.dump(era5_state, f)

### Maeve's figures

In [ ]:
import string
import pickle
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
import nicopal as ncp

temperature_cmap = ncp.pal("Boron")
temperature_difference_cmap = ncp.pal("Cobalt")

# =========================
# Settings & Data Setup
# =========================

sample_ds = list(datasets_results.values())[0]
LAT = sample_ds.latitude.values
LON = sample_ds.longitude.values

valid_date = pd.Timestamp("2026-05-25")
comparison_init_date = pd.Timestamp("2026-05-25")
panel_init_dates = ["2026051700", "2026051800"] 

g0 = 9.80665

# Target region
box_lon_min, box_lon_max = -5, 5.5
box_lat_min, box_lat_max = 44.5, 55

# Broad plotting region
map_lon_min, map_lon_max = -30, 25
map_lat_min, map_lat_max = 25, 65

# 1D Boolean Masks for spatial slicing
box_idx = (
    (LAT >= box_lat_min) & (LAT <= box_lat_max) & 
    (LON >= box_lon_min) & (LON <= box_lon_max)
)

map_idx = (
    (LAT >= map_lat_min) & (LAT <= map_lat_max) & 
    (LON >= map_lon_min) & (LON <= map_lon_max)
)

temperature_difference_limit = 6

# =========================
# Helper functions
# =========================

def target_box_mean(field):
    """Cosine-latitude-weighted target-region mean for 1D N320 data."""
    # Field is kept uncropped (48k length) until this mask is applied
    subset = field.isel(values=box_idx)
    weights = xr.DataArray(np.cos(np.deg2rad(LAT[box_idx])), dims=["values"])
    return subset.weighted(weights).mean(dim="values", skipna=True)

def add_map_features(ax):
    ax.set_extent(
        [map_lon_min, map_lon_max, map_lat_min, map_lat_max],
        crs=ccrs.PlateCarree(),
    )
    ax.coastlines(linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.add_patch(
        Rectangle(
            (box_lon_min, box_lat_min),
            box_lon_max - box_lon_min,
            box_lat_max - box_lat_min,
            fill=False, edgecolor="lime", linewidth=2,
            transform=ccrs.PlateCarree(), zorder=20,
        )
    )

def add_panel_label(ax, label):
    ax.text(
        0.02, 0.97, f"({label})",
        transform=ax.transAxes, ha="left", va="top",
        fontweight="bold", zorder=30,
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.75, pad=0.1),
    )

def plot_absolute_panel(ax, t2m_anom, z500_field, t_anom_limit, z500_levels, panel_label):
    # Centered anomaly colormap for the top panels
    anom_norm = mcolors.TwoSlopeNorm(vmin=-t_anom_limit, vcenter=0, vmax=t_anom_limit)
    
    # Slice the data to the map region right as it enters matplotlib
    temp_plot = ax.tripcolor(
        LON[map_idx], LAT[map_idx], t2m_anom.values[map_idx],
        cmap=temperature_difference_cmap, norm=anom_norm,
        shading="gouraud", transform=projection,
    )

    z_plot = ax.tricontour(
        LON[map_idx], LAT[map_idx], z500_field.values[map_idx],
        levels=z500_levels, colors="black", linewidths=1,
        transform=projection,
    )

    ax.clabel(z_plot, inline=True, fmt="%d")
    add_map_features(ax)
    add_panel_label(ax, panel_label)
    return temp_plot

def plot_difference_panel(ax, temperature_difference, z500_difference, legend_title, panel_label):
    diff_norm = mcolors.TwoSlopeNorm(
        vmin=-temperature_difference_limit, vcenter=0, vmax=temperature_difference_limit,
    )

    # Slice the data to the map region right as it enters matplotlib
    temp_plot = ax.tripcolor(
        LON[map_idx], LAT[map_idx], temperature_difference.values[map_idx],
        cmap=temperature_difference_cmap, norm=diff_norm,
        shading="gouraud", transform=projection,
    )

    z_plot = ax.tricontour(
        LON[map_idx], LAT[map_idx], z500_difference.values[map_idx],
        levels=10, colors="black", linewidths=1,
        transform=projection,
    )

    ax.clabel(z_plot, inline=True, fmt="%+d")

    # Fixed Dask evaluation here
    z_min = float(z500_difference.min(skipna=True).values)
    z_max = float(z500_difference.max(skipna=True).values)

    if z_min <= 0 <= z_max:
        ax.tricontour(
            LON[map_idx], LAT[map_idx], z500_difference.values[map_idx],
            levels=[0], colors="black", linewidths=2,
            transform=projection,
        )

    add_map_features(ax)
    add_panel_label(ax, panel_label)

    # Fixed Dask evaluation here
    target_t_difference = float(target_box_mean(temperature_difference).values)
    target_z_difference = float(target_box_mean(z500_difference).values)

    legend_handles = [
        Rectangle((0, 0), 1, 1, facecolor="lightcoral", edgecolor="black", label=f"T2m: {target_t_difference:+.2f} $^\circ$C"),
        Line2D([0], [0], color="black", linewidth=1.5, label=f"Z500: {target_z_difference:+.1f} m"),
    ]

    # Explicitly force the background box on to override any global styles
    ax.legend(
        handles=legend_handles, loc="lower right", title=legend_title,
        frameon=True, facecolor="white", edgecolor="black", framealpha=0.95, 
        fontsize="x-small", title_fontsize="x-small",
        borderpad=0.3, labelspacing=0.2, handlelength=1.2, handletextpad=0.4
    )
    return temp_plot

# =========================
# Load ERA5 Reference Data
# =========================

with open("/homedata/pchevali/mask_eu.pkl", "rb") as f:
    eu_mask = pickle.load(f)

era5_path = "/homedata/pchevali/AIFS_INPUTS/era5_daily_mean_20260525.pkl"
with open(era5_path, "rb") as f:
    era5_state = pickle.load(f)

era5_t2m_masked = era5_state["fields"]["2t"][-1][eu_mask] - 273.15
era5_z500_masked = era5_state["fields"]["z_500"][-1][eu_mask] / g0

# Keep them at the full 48k length
t2m_later_ensmean = xr.DataArray(era5_t2m_masked, dims=["values"])
z500_later_ensmean = xr.DataArray(era5_z500_masked, dims=["values"])

print(f"\nTarget-box ensemble-mean T2m, init {comparison_init_date:%Y-%m-%d}: {float(target_box_mean(t2m_later_ensmean)):.2f} $^\circ$C")
print(f"Target-box ensemble-mean Z500, init {comparison_init_date:%Y-%m-%d}: {float(target_box_mean(z500_later_ensmean)):.1f} m")

# =========================
# Member Selection Loop
# =========================

selection = {}

for panel_init_str in panel_init_dates:
    panel_init = pd.to_datetime(panel_init_str, format="%Y%m%d%H")
    v = datasets_results[panel_init_str]
    
    valid_dates = pd.to_datetime(v.valid_time.values).floor("D")
    idx_target = valid_dates == valid_date
    
    t2m_day = v["2t"].isel(step=idx_target).mean("step") - 273.15
    z500_day = v["z_500"].isel(step=idx_target).mean("step") / g0
    
    t2m_target_means = target_box_mean(t2m_day)
    z500_target_means = target_box_mean(z500_day)
    
    # Fixed Dask evaluation here
    member_high_t2m = int(t2m_target_means.idxmax(dim="number", skipna=True).values)
    member_high_z500 = int(z500_target_means.idxmax(dim="number", skipna=True).values)

    print(f"\nInitialization {panel_init:%Y-%m-%d}, valid {valid_date:%Y-%m-%d}")
    print(f"  Hottest target-box T2m member: {member_high_t2m}")
    print(f"  Highest target-box Z500 member: {member_high_z500}")

    # Save the full 48k arrays for plotting later
    selection[panel_init] = {
        "member_high_t2m": member_high_t2m,
        "member_high_z500": member_high_z500,
        "t_high_t2m": t2m_day.sel(number=member_high_t2m),
        "z_high_t2m": z500_day.sel(number=member_high_t2m),
        "t_high_z500": t2m_day.sel(number=member_high_z500),
        "z_high_z500": z500_day.sel(number=member_high_z500),
    }

# =========================
# Scaling & Plotting
# =========================

# Ensure climatology is in Celsius (if it was imported as raw ERA5 in Kelvin)
climato_mean = float(np.nanmean(climato_era5))
climato_c = climato_era5 - 273.15 if climato_mean > 150 else climato_era5

# Calculate anomaly limits for the top panels dynamically based on the subtracted fields
all_t_anoms = [
    selection[pd.to_datetime(d, format="%Y%m%d%H")][f"t_{m}"] - climato_c 
    for d in panel_init_dates for m in ["high_t2m", "high_z500"]
]
t_anom_limit = max(float(np.abs(field.isel(values=map_idx)).max(skipna=True).values) for field in all_t_anoms)
t_anom_limit = np.ceil(t_anom_limit)

all_z500_fields = [selection[pd.to_datetime(d, format="%Y%m%d%H")][f"z_{m}"] for d in panel_init_dates for m in ["high_t2m", "high_z500"]]
# Fixed Dask evaluation here
z500_min = min(float(field.isel(values=map_idx).min(skipna=True).values) for field in all_z500_fields)
z500_max = max(float(field.isel(values=map_idx).max(skipna=True).values) for field in all_z500_fields)
z500_levels = np.arange(np.floor(z500_min / 40) * 40, np.ceil(z500_max / 40) * 40 + 40, 40)

projection = ccrs.PlateCarree()

# Added a spacer row and slightly increased height_factor
fig = plt.figure(figsize=get_figsize(WIDTH_INSA, height_factor=1.5))

# nrows=7. The 4th item (index 3) is a spacer with height_ratio 0.2
gs = fig.add_gridspec(
    nrows=7, ncols=2, 
    height_ratios=[1, 1, 0.05, 0.05, 1, 1, 0.05], 
    hspace=0.15, wspace=0.05
)

# Map axes mapped to rows 1 and 2
axes_abs = [
    fig.add_subplot(gs[0, 0], projection=projection),
    fig.add_subplot(gs[0, 1], projection=projection),
    fig.add_subplot(gs[1, 0], projection=projection),
    fig.add_subplot(gs[1, 1], projection=projection)
]

# Colorbar for the absolute maps
cax_abs = fig.add_subplot(gs[2, :])

# ROW 3 is left intentionally blank to prevent the colorbar text from overlapping the maps below

# Difference axes mapped to rows 5 and 6
axes_diff = [
    fig.add_subplot(gs[4, 0], projection=projection),
    fig.add_subplot(gs[4, 1], projection=projection),
    fig.add_subplot(gs[5, 0], projection=projection),
    fig.add_subplot(gs[5, 1], projection=projection)
]

# Colorbar for the difference maps
cax_diff = fig.add_subplot(gs[6, :])

member_types = [("high_z500", "highest-Z500 member"), ("high_t2m", "hottest member")]
letters = string.ascii_lowercase

absolute_plot = None
difference_plot = None
print("\nPanel key:")

col = 0
for member_key, member_label in member_types:
    for panel_init_str in panel_init_dates:
        panel_init = pd.to_datetime(panel_init_str, format="%Y%m%d%H")
        data = selection[panel_init]

        member_num = data[f"member_{member_key}"]
        t_field = data[f"t_{member_key}"]
        z_field = data[f"z_{member_key}"]

        # Anomaly calculated for top panels
        t_anom = t_field - climato_c

        abs_label = letters[col]
        diff_label = letters[4 + col]

        print(f"  ({abs_label}) Member {member_num}: {member_label}, initialization {panel_init:%d %b %Y}")
        print(f"  ({diff_label}) Member {member_num} ({member_label}, init {panel_init:%d %b %Y}) − era5 ({comparison_init_date:%d %b})")

        panel_plot = plot_absolute_panel(axes_abs[col], t_anom, z_field, t_anom_limit, z500_levels, abs_label)
        if absolute_plot is None:
            absolute_plot = panel_plot

        t_diff = t_field - t2m_later_ensmean
        z_diff = z_field - z500_later_ensmean
        legend_title = f"member {member_num} $-$ era5"

        difference_plot = plot_difference_panel(axes_diff[col], t_diff, z_diff, legend_title, diff_label)
        col += 1

temperature_cbar = fig.colorbar(absolute_plot, cax=cax_abs, orientation="horizontal")
temperature_cbar.set_label("2 meter temperature anomaly vs climatology [$^\circ$C]")

difference_cbar = fig.colorbar(difference_plot, cax=cax_diff, orientation="horizontal")
difference_cbar.set_label(f"2 meter temperature difference: member $-$ era5 {comparison_init_date:%d %b} [$^\circ$C]")

plt.savefig("MAY_2026_HEATWAVE/z500_member_selection_pcolormesh.pdf", bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import nicopal as ncp

sequential_cmap = ncp.pal("Boron")
diverging_cmap = ncp.pal("Cobalt")

# =========================
# Combined figure:
#   row 1 -> ensemble plume (all members + ensemble mean per init date)
#   row 2 -> change in exceedance probability from one init to the next
#   row 3 -> across-member std of box-mean z500 valid on target_valid_date,
#            as a function of forecast initialization date
#   row 4 -> ensemble-mean z500 maps for the chosen initializations
# =========================

# =========================
# Settings & Data Setup
# =========================

# Get coordinates from the first dataset
sample_ds = list(datasets_results.values())[0]
LAT = sample_ds.latitude.values
LON = sample_ds.longitude.values
g0 = 9.80665

box_lon_min, box_lon_max = -5, 5.5
box_lat_min, box_lat_max = 44.5, 55

map_lon_min, map_lon_max = -30, 25
map_lat_min, map_lat_max = 25, 65

threshold_geopotential = 57500.0  
threshold = threshold_geopotential / g0  

init_start = pd.Timestamp("2026-05-15")
plot_start = pd.Timestamp("2026-05-15")
plot_end = pd.Timestamp("2026-06-05")
highlight_init_date = pd.Timestamp("2026-05-18")

target_valid_date = pd.Timestamp("2026-05-25")
init_dates_to_plot = [
    pd.Timestamp("2026-05-17"),
    pd.Timestamp("2026-05-18"),
    pd.Timestamp("2026-05-19"),
]

# 1D N320 boolean masks
box_idx = (
    (LAT >= box_lat_min) & (LAT <= box_lat_max) & 
    (LON >= box_lon_min) & (LON <= box_lon_max)
)

map_idx = (
    (LAT >= map_lat_min) & (LAT <= map_lat_max) & 
    (LON >= map_lon_min) & (LON <= map_lon_max)
)

# =========================
# Big-box regional mean & Exceedance Probability (Rows 1 & 2)
# =========================

plume_data = {}
prob_df_list = []

for date_str, v in datasets_results.items():
    init_dt = pd.to_datetime(date_str, format="%Y%m%d%H")
    if init_dt < init_start: continue
        
    z_box = v["z_500"].isel(values=box_idx) / g0
    weights = xr.DataArray(np.cos(np.deg2rad(LAT[box_idx])), dims=["values"])
    z500_mean = z_box.weighted(weights).mean(dim="values", skipna=True)
    
    plume_data[init_dt] = {
        "mean_z": z500_mean,
        "valid_time": pd.to_datetime(v.valid_time.values)
    }
    
    prob = (z500_mean > threshold).mean(dim="number", skipna=True) * 100
    prob_df_list.append(pd.DataFrame({
        "init_time": init_dt,
        "valid_time": plume_data[init_dt]["valid_time"],
        "prob": prob.values
    }))

prob_df = pd.concat(prob_df_list, ignore_index=True)
init_dates = pd.DatetimeIndex(list(plume_data.keys())).sort_values()

prob_wide = prob_df.pivot_table(index="init_time", columns="valid_time", values="prob")
dprob_wide = prob_wide.diff(axis=0).iloc[1:]

dprob_x_dates = pd.to_datetime(dprob_wide.columns)
dprob_y_dates = pd.to_datetime(dprob_wide.index)

dprob_x = np.repeat(mdates.date2num(dprob_x_dates.to_pydatetime())[None, :], len(dprob_y_dates), axis=0)
dprob_y = np.repeat(mdates.date2num(dprob_y_dates.to_pydatetime())[:, None], len(dprob_x_dates), axis=1)

dprob_abs_max = float(np.nanmax(np.abs(dprob_wide.values)))
if not np.isfinite(dprob_abs_max) or dprob_abs_max == 0:
    dprob_abs_max = 1.0

dprob_norm = mcolors.TwoSlopeNorm(vmin=-dprob_abs_max, vcenter=0, vmax=dprob_abs_max)

print("Probability-change color scale: +/-", f"{dprob_abs_max:.1f}", "percentage points")
print("Maximum probability increase:", np.nanmax(dprob_wide.values), "percentage points")
print("Maximum probability decrease:", np.nanmin(dprob_wide.values), "percentage points")

# =========================
# Across-member standard deviation of the box-mean z500 (Row 3)
# =========================

std_x_inits = []
std_values = []
frac_values = []

for init_date in init_dates:
    fcst = plume_data[init_date]["mean_z"]
    fcst_time = pd.to_datetime(plume_data[init_date]["valid_time"])
    
    # Find step index for target_valid_date
    idx = np.where(fcst_time.floor("D") == target_valid_date.floor("D"))[0]
    
    if len(idx) > 0:
        z_target_day = fcst.isel(step=idx).mean("step")
        std_val = float(z_target_day.std(dim="number", skipna=True).values)
        frac_val = float((z_target_day > threshold).mean(dim="number", skipna=True).values)
        
        std_x_inits.append(init_date)
        std_values.append(std_val)
        frac_values.append(frac_val)

std_x_inits = pd.to_datetime(std_x_inits)
std_values = np.array(std_values)
frac_values = np.array(frac_values)

if len(std_values) > 0:
    print(
        "Across-member std valid", target_valid_date.date(),
        "ranges from", f"{np.nanmin(std_values):.1f}", "to",
        f"{np.nanmax(std_values):.1f}", "m over", len(std_values), "initializations"
    )

# =========================
# Ensemble-mean maps for the chosen initializations (Row 4)
# =========================

maps = []
for init_str in init_dates_to_plot:
    v = datasets_results[init_str.strftime("%Y%m%d%H")]
    valid_dates = pd.to_datetime(v.valid_time.values).floor("D")
    idx_target = valid_dates == target_valid_date
    
    z_valid = v["z_500"].isel(step=idx_target).mean(["step", "number"]) / g0
    maps.append(z_valid)

all_map_values = xr.concat([m.isel(values=map_idx) for m in maps], dim="case")

map_vmin = float(all_map_values.min().values)
map_vmax = float(all_map_values.max().values)

n_fill_levels = 35
n_line_levels = 25
map_fill_levels = np.linspace(map_vmin, map_vmax, n_fill_levels)
map_line_levels = np.linspace(map_vmin, map_vmax, n_line_levels)

# =========================
# Quick diagnostics: peak z500 valid on target_valid_date
# =========================

raw_max = -np.inf
raw_max_init = None

for date_str, v in datasets_results.items():
    init_dt = pd.to_datetime(date_str, format="%Y%m%d%H")
    valid_dates = pd.to_datetime(v.valid_time.values).floor("D")
    idx_target = valid_dates == target_valid_date
    
    if not idx_target.any(): continue
        
    field = v["z_500"].isel(step=idx_target).mean("step") / g0
    field_max = float(field.isel(values=map_idx).max().values)
    
    if field_max > raw_max:
        raw_max = field_max
        raw_max_init = init_dt

if np.isfinite(raw_max):
    print(
        f"Max z500 valid {target_valid_date.date()} "
        f"(any member, any init, map region): {raw_max:.1f} m "
        f"(peak from init {raw_max_init.date()})"
    )
else:
    print(f"No forecast valid on {target_valid_date.date()} among available initializations.")

ensmean_init_str = target_valid_date.strftime("%Y%m%d00")
if ensmean_init_str in datasets_results:
    v = datasets_results[ensmean_init_str]
    valid_dates = pd.to_datetime(v.valid_time.values).floor("D")
    idx_target = valid_dates == target_valid_date
    if idx_target.any():
        ensmean_init_field = v["z_500"].isel(step=idx_target).mean(["step", "number"]) / g0
        ensmean_init_max = float(ensmean_init_field.isel(values=map_idx).max().values)
        print(f"Max z500 of ensemble mean initialized {target_valid_date.date()} (map region): {ensmean_init_max:.1f} m")
else:
    print(f"No initialization on {target_valid_date.date()} in the dataset, so ensemble-mean max is unavailable.")

# =========================
# Combined figure Plotting
# =========================

def add_panel_label(ax, label):
    ax.text(
        0.02, 0.97, f"({label})",
        transform=ax.transAxes, ha="left", va="top",
        fontweight="bold", zorder=30,
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.75, pad=2),
    )

# Uses get_figsize with WIDTH_INSA and the requested 15/10 aspect ratio
fig = plt.figure(figsize=get_figsize(WIDTH_INSA, height_factor=16/10))
gs = fig.add_gridspec(
    nrows=4, ncols=4, width_ratios=[1, 1, 1, 0.035], 
    height_ratios=[1, 1.1, 0.8, 1.1], hspace=0.55, wspace=0.1
)

ax_top = fig.add_subplot(gs[0, 0:3])
cax_top = fig.add_subplot(gs[0, 3])
ax_mid = fig.add_subplot(gs[1, 0:3])
cax_mid = fig.add_subplot(gs[1, 3])
ax_std = fig.add_subplot(gs[2, 0:3])

# Create only as many map subplots as we actually have dates available
map_axes = [fig.add_subplot(gs[3, i], projection=ccrs.PlateCarree()) for i in range(len(init_dates_to_plot))]
cax_bottom = fig.add_subplot(gs[3, 3])

# --- Row 1: ensemble plume ---
plume_colors = ncp.pal_sample("Osmium", len(init_dates))

for i, init_date in enumerate(init_dates):
    fcst = plume_data[init_date]["mean_z"]
    fcst_time = plume_data[init_date]["valid_time"]
    color = plume_colors[i]

    for member in fcst["number"].values:
        ax_top.plot(fcst_time, fcst.sel(number=member).values, color=color, alpha=0.22, linewidth=0.8)
    ax_top.plot(fcst_time, fcst.mean("number", skipna=True).values, color=color, linewidth=2.2)

threshold_line = ax_top.axhline(threshold, color="black", linewidth=2.0, linestyle="--", label=f"{threshold:.0f} m threshold")

ax_top.set_ylabel("Box z500 mean [m]")
ax_top.set_xlabel("Forecast valid date")
add_panel_label(ax_top, "a")
ax_top.set_xlim(plot_start, plot_end)

daily_ticks = pd.date_range(start=plot_start, end=plot_end, freq="1D")
ax_top.set_xticks(daily_ticks)
ax_top.set_xticklabels(daily_ticks.strftime("%d %b"), rotation=90, ha="center")
ax_top.grid(True, alpha=0.25)
ax_top.legend(handles=[threshold_line], loc="lower left")

plume_cmap = mcolors.ListedColormap(plume_colors)
plume_boundaries = np.arange(len(init_dates) + 1) - 0.5
plume_norm = mcolors.BoundaryNorm(plume_boundaries, plume_cmap.N)

cbar_top = fig.colorbar(plt.cm.ScalarMappable(norm=plume_norm, cmap=plume_cmap), cax=cax_top)
cbar_top.set_ticks(np.arange(len(init_dates)))
cbar_top.set_ticklabels(init_dates.strftime("%m-%d"))
cbar_top.set_label("Initialization date")

# --- Row 2: change in exceedance probability ---
pcm_mid = ax_mid.pcolormesh(dprob_x, dprob_y, dprob_wide.values, shading="nearest", cmap=diverging_cmap, norm=dprob_norm)
ax_mid.axhline(mdates.date2num(highlight_init_date), color="black", linewidth=2.5, linestyle="-", zorder=20)
ax_mid.set_ylabel("Current initialization date")
ax_mid.set_xlabel("Forecast valid date")
add_panel_label(ax_mid, "b")

ax_mid.set_yticks(mdates.date2num(dprob_y_dates.to_pydatetime()))
ax_mid.set_yticklabels(dprob_y_dates.strftime("%m-%d"))
ax_mid.set_xlim(plot_start, plot_end)
ax_mid.set_xticks(daily_ticks)
ax_mid.set_xticklabels(daily_ticks.strftime("%d %b"), rotation=90, ha="center")
ax_mid.grid(True, alpha=0.25, color="white", linewidth=0.6, zorder=25)

cbar_mid = fig.colorbar(pcm_mid, cax=cax_mid)
cbar_mid.set_label("Probability change [%]")

# --- Row 3: across-member std & fraction exceeding ---
frac_color = "#c1121f"

line_std, = ax_std.plot(
    std_x_inits, std_values, color="black", linewidth=2.0, marker="o", markersize=5, label="Across-member std",
)
ax_std.set_ylabel("Across-member std\nof box z500 [m]")
ax_std.set_xlabel("Forecast initialization date")
#ax_std.set_title(f"Ensemble spread of box-mean z500 valid {target_valid_date.strftime('%d %b %Y')}")
add_panel_label(ax_std, "c")
ax_std.grid(True, alpha=0.25)

ax_frac = ax_std.twinx()
line_frac, = ax_frac.plot(
    std_x_inits, frac_values, color=frac_color, linewidth=2.0, linestyle="--", marker="s", markersize=5, label=f"Fraction > {threshold:.0f} m",
)
ax_frac.set_ylabel("Fraction of members\nexceeding threshold", color=frac_color)
ax_frac.tick_params(axis="y", labelcolor=frac_color)
ax_frac.set_ylim(-0.02, 1.02)

ax_std.legend(handles=[line_std, line_frac], loc="best")

if len(std_x_inits) > 0:
    std_ticks = pd.date_range(start=std_x_inits.min(), end=std_x_inits.max(), freq="1D")
    ax_std.set_xticks(std_ticks)
    ax_std.set_xticklabels(std_ticks.strftime("%d %b"), rotation=90, ha="center")

# --- Row 4: ensemble-mean z500 maps ---
map_labels = ["d", "e", "f"]
cf_bottom = None

# Custom formatters for perfect LaTeX rendering of degrees, avoiding Cartopy Unicode issues
def format_lon_latex(x, pos):
    val = int(x) if float(x).is_integer() else round(x, 1)
    if val < 0:
        return fr"${-val}^{{\circ}}\mathrm{{W}}$"
    elif val > 0 and val < 180:
        return fr"${val}^{{\circ}}\mathrm{{E}}$"
    else:
        return fr"${val}^{{\circ}}$"

def format_lat_latex(x, pos):
    val = int(x) if float(x).is_integer() else round(x, 1)
    if val < 0:
        return fr"${-val}^{{\circ}}\mathrm{{S}}$"
    elif val > 0:
        return fr"${val}^{{\circ}}\mathrm{{N}}$"
    else:
        return fr"${val}^{{\circ}}$"

for i, (ax, z_map, init_date, panel_label) in enumerate(zip(map_axes, maps, init_dates_to_plot, map_labels)):
    ax.set_extent([map_lon_min, map_lon_max, map_lat_min, map_lat_max], crs=ccrs.PlateCarree())
    ax.coastlines(resolution="10m", linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.add_feature(cfeature.LAND, alpha=0.25)
    ax.add_feature(cfeature.OCEAN, alpha=0.15)

    cf_bottom = ax.tricontourf(
        LON[map_idx], LAT[map_idx], z_map.values[map_idx],
        levels=map_fill_levels, cmap=sequential_cmap, extend="both", transform=ccrs.PlateCarree(),
    )

    cs = ax.tricontour(
        LON[map_idx], LAT[map_idx], z_map.values[map_idx],
        levels=map_line_levels, colors="black", linewidths=0.5, alpha=0.6, transform=ccrs.PlateCarree(),
    )

    ax.clabel(cs, inline=True, fmt="%.0f")
    ax.plot(
        [box_lon_min, box_lon_max, box_lon_max, box_lon_min, box_lon_min],
        [box_lat_min, box_lat_min, box_lat_max, box_lat_max, box_lat_min],
        color="red", linewidth=2, transform=ccrs.PlateCarree(),
    )

    add_panel_label(ax, panel_label)
    gl = ax.gridlines(draw_labels=True, linestyle="--", alpha=0.4)
    gl.top_labels = False
    gl.right_labels = False
    
    # Assign the custom LaTeX formatters to the gridlines
    gl.xformatter = mticker.FuncFormatter(format_lon_latex)
    gl.yformatter = mticker.FuncFormatter(format_lat_latex)
    
    if i > 0:
        gl.left_labels = False

if cf_bottom:
    cbar_bottom = fig.colorbar(cf_bottom, cax=cax_bottom)
    cbar_bottom.set_label("Ensemble mean z500 [m]")

plt.tight_layout()
plt.savefig("MAY_2026_HEATWAVE/500_combined_overview2.pdf", bbox_inches="tight")
plt.show()

## Cutoff low detection

#### Compute

In [ ]:
#setup
base_dir = "/scratchx/pchevali/MAY_2026_HEATWAVE_PROCESSED"
target_var = "z_500"

# peninsula box
lat_min, lat_max = 37.0, 45.0
lon_min, lon_max = -17.0, -2.5
lat_min, lat_max = 25.0, 50.0
lon_min, lon_max = -35.0, 0.0

In [ ]:
forecast_init_date = "2026051700"
detect_cutoffs(base_dir, forecast_init_date, target_var, lat_min, lat_max, lon_min, lon_max)

In [ ]:
forecast_init_date = "2026051800"
detect_cutoffs(base_dir, forecast_init_date, target_var, lat_min, lat_max, lon_min, lon_max)

In [ ]:
# forecast_init_date = "2026051900"
# detect_cutoffs(base_dir, forecast_init_date, target_var, lat_min, lat_max, lon_min, lon_max)

#### Load results

In [ ]:
lat_min, lat_max = 37.0, 45.0
lon_min, lon_max = -17.0, -2.5

In [ ]:
date_25 = np.datetime64("2026-05-25T12:00")

labels = []
probs_25 = []
probs_both = []
members_exceeding = {}

for k, v in datasets_results.items():
    labels.append(pd.to_datetime(k, format="%Y%m%d%H%M").strftime("%b %d"))
    z500_mean = v["z_500"].isel(values=average_region).mean("values")
    idx_25 = v.valid_time.values == date_25
    mask_25 = (z500_mean.isel(step=idx_25) > 57500).values.squeeze()
    prob_25 = np.mean(mask_25)
    probs_25.append(prob_25)
    members_exceeding[k] = v["number"].values[mask_25]

In [ ]:
def extract_cutoff_members(init_date, target_date="2026-05-25", exceeding=True):
    """Filters cols_index CSVs and returns members with/without low pressure areas."""
    df = pd.read_csv(f"cols_index_Init_{init_date}.csv")

    if exceeding:
        valid_members = members_exceeding[init_date]
    else:
        valid_members = datasets_results[init_date].number.values

    mask = (
        (df["Target_Date"] == target_date)
        & (df["up_or_down"] == "up")
        & (df["lat"].between(lat_min, lat_max, inclusive="neither"))
        & (df["lon"].between(lon_min, lon_max, inclusive="neither"))
        & (df["Member"].isin(valid_members))
    )

    with_lp = df[mask].sort_values("So", ascending=False)["Member"].unique()
    without_lp = np.setdiff1d(valid_members, with_lp)

    return with_lp, without_lp, with_lp[:10]


def extract_d4_members(init_date, exceeding=True):
    """Filters COL_d4 CSVs and returns members with/without high D4."""
    df = pd.read_csv(f"COL_d4_index_Init_{init_date}_UP.csv")

    if exceeding:
        valid_members = members_exceeding[init_date]
    else:
        valid_members = datasets_results[init_date].number.values

    mask = df["Member"].isin(valid_members) & (df["Total_So"] > 0)

    with_d4 = df[mask].sort_values("Total_So", ascending=False)["Member"].unique()
    without_d4 = np.setdiff1d(valid_members, with_d4)

    return with_d4, without_d4, with_d4[:10]

In [ ]:
# ==========================================
# Exceeding Threshold Only
# ==========================================

# 17th Init
(
    members_with_low_pressure_area_17th,
    members_without_low_pressure_area_17th,
    members_with_low_pressure_area_17th_top10,
) = extract_cutoff_members("2026051700", exceeding=True)

(
    members_with_high_D4_17th,
    members_without_high_D4_17th,
    members_with_high_D4_17th_top10,
) = extract_d4_members("2026051700", exceeding=True)

# 18th Init
(
    members_with_low_pressure_area_18th,
    members_without_low_pressure_area_18th,
    members_with_low_pressure_area_18th_top10,
) = extract_cutoff_members("2026051800", exceeding=True)

(
    members_with_high_D4_18th,
    members_without_high_D4_18th,
    members_with_high_D4_18th_top10,
) = extract_d4_members("2026051800", exceeding=True)


# ==========================================
# No Threshold (All members)
# ==========================================

# 17th Init
(
    members_with_low_pressure_area_17th_all,
    members_without_low_pressure_area_17th_all,
    members_with_low_pressure_area_17th_all_top10,
) = extract_cutoff_members("2026051700", exceeding=False)

(
    members_with_high_D4_17th_all,
    members_without_high_D4_17th_all,
    members_with_high_D4_17th_all_top10,
) = extract_d4_members("2026051700", exceeding=False)

# 18th Init
(
    members_with_low_pressure_area_18th_all,
    members_without_low_pressure_area_18th_all,
    members_with_low_pressure_area_18th_all_top10,
) = extract_cutoff_members("2026051800", exceeding=False)

(
    members_with_high_D4_18th_all,
    members_without_high_D4_18th_all,
    members_with_high_D4_18th_all_top10,
) = extract_d4_members("2026051800", exceeding=False)

## Composite diagnostics

### 10 hottest vs 10 coldest

In [ ]:
target_dates = [
    np.datetime64("2026-05-25T12:00")
]  # , np.datetime64("2026-05-26T12:00")]

# temperatures and their pressure levels
t_vars = [
    "t_1000",
    "t_925",
    "t_850",
    "t_700",
    "t_600",
    "t_500",
    "t_400",
    "t_300",
    "t_250",
    "t_200",
]
levels = [1000, 925, 850, 700, 600, 500, 400, 300, 250, 200]
subset_size = 10

# loop for each forecast starting date
for k, v in datasets_results.items():
    if k in ["2026051700", "2026051800"]:
        exceeding_members = members_exceeding[k]
        time_mask = np.isin(v.valid_time.values, target_dates)

        # sorting the surface temperatures
        t2m_exceeding = (
            v["2t"]
            .isel(values=average_region)
            .isel(step=time_mask)
            .sel(number=exceeding_members)
        )
        t2m_mean_per_member = t2m_exceeding.mean(dim=["values", "step"])
        t2m_sorted = t2m_mean_per_member.sortby(t2m_mean_per_member, ascending=False)

        hot_members = t2m_sorted.number[:subset_size].values
        cold_members = t2m_sorted.number[-subset_size:].values

        ## COMPUTE DATA FOR THE 4 PLOTS

        # composites of geopt
        z500_hot_comp = (
            v["z_500"]
            .isel(values=plot_region_indexs)
            .isel(step=time_mask)
            .sel(number=hot_members)
            .mean(dim=["step", "number"])
            .values
        ) / 9.81
        z500_cold_comp = (
            v["z_500"]
            .isel(values=plot_region_indexs)
            .isel(step=time_mask)
            .sel(number=cold_members)
            .mean(dim=["step", "number"])
            .values
        ) / 9.81

        # surface temperature difference
        t2m_hot_map = (
            v["2t"]
            .isel(values=plot_region_indexs)
            .isel(step=time_mask)
            .sel(number=hot_members)
            .mean(dim=["step", "number"])
            .values
        )
        t2m_hot_map_anom = t2m_hot_map - climato_era5[plot_region_indexs]
        t2m_cold_map = (
            v["2t"]
            .isel(values=plot_region_indexs)
            .isel(step=time_mask)
            .sel(number=cold_members)
            .mean(dim=["step", "number"])
            .values
        )
        t2m_cold_map_anom = t2m_cold_map - climato_era5[plot_region_indexs]
        t2m_diff_map = t2m_hot_map - t2m_cold_map

        # vertical profile of temperature difference

        profile_dates = [
            np.datetime64("2026-05-25T12:00"),
            # np.datetime64("2026-05-26T12:00"),
            # np.datetime64("2026-05-27T12:00"),
            # np.datetime64("2026-05-28T12:00"),
            # np.datetime64("2026-05-29T12:00"),
            # np.datetime64("2026-05-30T12:00"),
        ]

        profiles = {}
        for p_date in profile_dates:
            p_time_mask = np.isin(v.valid_time.values, [p_date])
            t_prof_diff = []
            for var in t_vars:
                # mean over space, time, and the members for the pressure level
                t_hot = (
                    v[var]
                    .isel(values=average_region)
                    .isel(step=p_time_mask)
                    .sel(number=hot_members)
                    .mean(dim=["values", "step", "number"])
                    .values
                )
                t_cold = (
                    v[var]
                    .isel(values=average_region)
                    .isel(step=p_time_mask)
                    .sel(number=cold_members)
                    .mean(dim=["values", "step", "number"])
                    .values
                )
                t_prof_diff.append(t_hot - t_cold)
            profiles[p_date] = t_prof_diff

        ## PLOT

        fig = plt.figure(
            figsize=get_figsize(WIDTH_INSA, fraction=1, height_factor=0.65),
            #constrained_layout=True,
        )

        z500_min = min(z500_hot_comp.min(), z500_cold_comp.min())
        z500_max = max(z500_hot_comp.max(), z500_cold_comp.max())

        levels_line = mtickers.MaxNLocator(nbins=13).tick_values(z500_min, z500_max)
        t2m_vmax = max(abs(t2m_hot_map_anom.min()), abs(t2m_hot_map_anom.max()))
        levels_fill = np.linspace(-t2m_vmax, t2m_vmax, 16)

        ax1 = fig.add_subplot(2, 2, 1, projection=ccrs.PlateCarree())

        cf1 = ax1.tricontourf(
            LON[plot_region_indexs],
            LAT[plot_region_indexs],
            t2m_hot_map_anom,
            levels=levels_fill,
            cmap=ncp.pal("Cobalt"),
        )
        contours1 = ax1.tricontour(
            LON[plot_region_indexs],
            LAT[plot_region_indexs],
            z500_hot_comp,
            levels=levels_line,
            colors="black",
        )
        ax1.clabel(contours1, fmt="%.0f", inline=True)
        ax1.coastlines()
        ax1.set_title(
            f"(a)",  # Composites of z_500 for the {subset_size} hottest members",
        )
        cbar1 = fig.colorbar(
            cf1,
            ax=ax1,
            orientation="vertical",
            shrink=0.7,
            ticks=mtickers.MaxNLocator(integer=True),
        )
        cbar1.set_label("Temperature anomaly ($^\circ$C)")

        # plot z500 composite for coldest members
        ax2 = fig.add_subplot(2, 2, 2, projection=ccrs.PlateCarree())
        cf2 = ax2.tricontourf(
            LON[plot_region_indexs],
            LAT[plot_region_indexs],
            t2m_cold_map_anom,
            levels=levels_fill,
            cmap=ncp.pal("Cobalt"),
        )
        contours2 = ax2.tricontour(
            LON[plot_region_indexs],
            LAT[plot_region_indexs],
            z500_cold_comp,
            levels=levels_line,
            colors="black",
        )
        ax2.clabel(contours2, fmt="%.0f", inline=True)
        ax2.coastlines()
        ax2.set_title(
            f"(b)",  # Composites of z_500 for the {subset_size} coldest members",
        )
        cbar2 = fig.colorbar(
            cf2,
            ax=ax2,
            orientation="vertical",
            shrink=0.7,
            ticks=mtickers.MaxNLocator(integer=True),
        )
        cbar2.set_label("Temperature anomaly ($^\circ$C)")

        # plot the difference in surface temperature
        ax3 = fig.add_subplot(2, 2, 3, projection=ccrs.PlateCarree())
        t2m_vmax = max(abs(t2m_diff_map.min()), abs(t2m_diff_map.max()))
        cf3 = ax3.tricontourf(
            LON[plot_region_indexs],
            LAT[plot_region_indexs],
            t2m_diff_map,
            levels=12,
            cmap=ncp.pal("Cobalt"),
            vmin=-t2m_vmax,
            vmax=t2m_vmax,
        )
        contours3 = ax3.tricontour(
            LON[plot_region_indexs],
            LAT[plot_region_indexs],
            z500_hot_comp - z500_cold_comp,
            levels=12,
            colors="black",
        )
        ax3.clabel(contours3, fmt="%.0f", inline=True)
        ax3.coastlines()

        lon_min, lon_max = average_longitude_bnd.start, average_longitude_bnd.stop
        lat_min, lat_max = average_latitude_bnd.start, average_latitude_bnd.stop
        ax3.plot(
            [lon_min, lon_max, lon_max, lon_min, lon_min],
            [lat_min, lat_min, lat_max, lat_max, lat_min],
            color="green",
            linewidth=2,
            transform=ccrs.PlateCarree(),
        )

        ax3.set_title(
            "(c)",  # Surface temperature difference (hot-cold)
        )
        cbar3 = fig.colorbar(
            cf3,
            ax=ax3,
            orientation="vertical",
            shrink=0.7,
            ticks=mtickers.MaxNLocator(integer=True),
        )
        cbar3.set_label("Temperature difference ($^\circ$C)")

        # plot vertical profile of temperature difference
        ax4 = fig.add_subplot(2, 2, 4)
        lon_range = np.max(LON[plot_region_indexs]) - np.min(LON[plot_region_indexs])
        lat_range = np.max(LAT[plot_region_indexs]) - np.min(LAT[plot_region_indexs])
        ax4.set_box_aspect(lat_range / lon_range)
        for i, (p_date, prof) in enumerate(profiles.items()):
            day_str = pd.to_datetime(p_date).strftime("%b %d")
            ax4.plot(
                prof,
                levels,
                marker="o",
                #linewidth=2,
                label=f"{day_str}",
            )
        ax4.axvline(0, color="black", linestyle="--", linewidth=1)
        ax4.set_ylabel("Pressure level (hPa)")
        ax4.set_xlabel("Temperature difference ($^\circ$C)")
        ax4.set_title(
            "(d)",  # Vertical profile of temperature differences (averaged over region)",
        )
        ax4.yaxis.tick_right()
        ax4.yaxis.set_label_position("right")
        ax4.grid(linestyle=":")
        ax4.invert_yaxis()  # invert axis so the ground is on the bottom
        ax4.legend(loc="best")
        ax4.tick_params(axis="both")
        #fake colorbar so it's aligned lol
        cbar4 = fig.colorbar(
            cf3,
            ax=ax4, 
            orientation="vertical", 
            shrink=0.7
        )
        cbar4.ax.set_visible(False)


        plt.tight_layout()
        plt.savefig(
            f"MAY_2026_HEATWAVE/diagnostic_hot_cold_25th_forecast_starting_{k}_{subset_size}_members.pdf",
            bbox_inches="tight",
        )

        plt.show()

        print(
            "-----------------------------------------------------------------------------------------------------------------------------"
        )

### with vs without cutoff

In [ ]:
target_dates = [
    np.datetime64("2026-05-25T12:00")
]  # , np.datetime64("2026-05-26T12:00")]

# temperatures and their pressure levels
t_vars = [
    "t_1000",
    "t_925",
    "t_850",
    "t_700",
    "t_600",
    "t_500",
    "t_400",
    "t_300",
    "t_250",
    "t_200",
]
levels = [1000, 925, 850, 700, 600, 500, 400, 300, 250, 200]
subset_size = 10

# loop for each forecast starting date
for k, v in datasets_results.items():
    if k in ["2026051700", "2026051800"]:
        time_mask = np.isin(v.valid_time.values, target_dates)

        hot_members = members_with_low_pressure_area_top10[k]
        cold_members = members_without_low_pressure_area[k]

        ## COMPUTE DATA FOR THE 4 PLOTS

        # composites of geopt
        z500_hot_comp = (
            v["z_500"]
            .isel(values=plot_region_indexs)
            .isel(step=time_mask)
            .sel(number=hot_members)
            .mean(dim=["step", "number"])
            .values
        ) / 9.81
        z500_cold_comp = (
            v["z_500"]
            .isel(values=plot_region_indexs)
            .isel(step=time_mask)
            .sel(number=cold_members)
            .mean(dim=["step", "number"])
            .values
        ) / 9.81

        # surface temperature difference
        t2m_hot_map = (
            v["2t"]
            .isel(values=plot_region_indexs)
            .isel(step=time_mask)
            .sel(number=hot_members)
            .mean(dim=["step", "number"])
            .values
        )
        t2m_hot_map_anom = t2m_hot_map - climato_era5[plot_region_indexs]
        t2m_cold_map = (
            v["2t"]
            .isel(values=plot_region_indexs)
            .isel(step=time_mask)
            .sel(number=cold_members)
            .mean(dim=["step", "number"])
            .values
        )
        t2m_cold_map_anom = t2m_cold_map - climato_era5[plot_region_indexs]
        t2m_diff_map = t2m_hot_map - t2m_cold_map

        # vertical profile of temperature difference
        profile_dates = [
            np.datetime64("2026-05-25T12:00"),
            # np.datetime64("2026-05-26T12:00"),
            # np.datetime64("2026-05-27T12:00"),
            # np.datetime64("2026-05-28T12:00"),
            # np.datetime64("2026-05-29T12:00"),
            # np.datetime64("2026-05-30T12:00"),
        ]

        profiles = {}
        for p_date in profile_dates:
            p_time_mask = np.isin(v.valid_time.values, [p_date])
            t_prof_diff = []
            for var in t_vars:
                # mean over space, time, and the members for the pressure level
                t_hot = (
                    v[var]
                    .isel(values=average_region)
                    .isel(step=p_time_mask)
                    .sel(number=hot_members)
                    .mean(dim=["values", "step", "number"])
                    .values
                )
                t_cold = (
                    v[var]
                    .isel(values=average_region)
                    .isel(step=p_time_mask)
                    .sel(number=cold_members)
                    .mean(dim=["values", "step", "number"])
                    .values
                )
                t_prof_diff.append(t_hot - t_cold)
            profiles[p_date] = t_prof_diff

        ## PLOT

        fig = plt.figure(
            figsize=get_figsize(WIDTH_INSA, fraction=1, height_factor=0.75),
            #constrained_layout=True,
        )
        # plt.suptitle(
        #     f"Dianostic for the 25th (averaged) \nForecast Initialization: {k}",
        #     $^\circ$C
        # )

        # plot z500 composite for hottest members

        z500_min = min(z500_hot_comp.min(), z500_cold_comp.min())
        z500_max = max(z500_hot_comp.max(), z500_cold_comp.max())

        levels_line = mtickers.MaxNLocator(nbins=13).tick_values(z500_min, z500_max)
        t2m_vmax = np.ceil(
            max(
                abs(t2m_hot_map_anom.min()),
                abs(t2m_hot_map_anom.max()),
                abs(t2m_cold_map_anom.min()),
                abs(t2m_cold_map_anom.max()),
            )
        )
        levels_fill = np.linspace(-t2m_vmax, t2m_vmax, 17)

        ax1 = fig.add_subplot(2, 2, 1, projection=ccrs.PlateCarree())

        cf1 = ax1.tricontourf(
            LON[plot_region_indexs],
            LAT[plot_region_indexs],
            t2m_hot_map_anom,
            levels=levels_fill,
            vmin=-t2m_vmax,
            vmax=t2m_vmax,
            cmap=ncp.pal("Cobalt"),
        )
        contours1 = ax1.tricontour(
            LON[plot_region_indexs],
            LAT[plot_region_indexs],
            z500_hot_comp,
            levels=levels_line,
            colors="black",
        )
        ax1.clabel(contours1, fmt="%.0f", inline=True)
        ax1.coastlines()
        ax1.set_title(
            f"(a)",  # Composites of z_500 for the {subset_size} members\nwith most intense cutoff",
        )
        cbar1 = fig.colorbar(
            cf1,
            ax=ax1,
            orientation="vertical",
            shrink=0.7,
            ticks=mtickers.MaxNLocator(integer=True),
        )
        cbar1.set_label("Temperature anomaly ($^\circ$C)")

        # plot z500 composite for coldest members
        ax2 = fig.add_subplot(2, 2, 2, projection=ccrs.PlateCarree())
        cf2 = ax2.tricontourf(
            LON[plot_region_indexs],
            LAT[plot_region_indexs],
            t2m_cold_map_anom,
            levels=levels_fill,
            vmin=-t2m_vmax,
            vmax=t2m_vmax,
            cmap=ncp.pal("Cobalt"),
        )
        contours2 = ax2.tricontour(
            LON[plot_region_indexs],
            LAT[plot_region_indexs],
            z500_cold_comp,
            levels=levels_line,
            colors="black",
        )
        ax2.clabel(contours2, fmt="%.0f", inline=True)
        ax2.coastlines()
        ax2.set_title(
            f"(b)",  # Composites of z_500 for the members without cutoff",
        )
        cbar2 = fig.colorbar(
            cf2,
            ax=ax2,
            orientation="vertical",
            shrink=0.7,
            ticks=mtickers.MaxNLocator(integer=True),
        )
        cbar2.set_label("Temperature anomaly ($^\circ$C)")

        # plot the difference in surface temperature
        ax3 = fig.add_subplot(2, 2, 3, projection=ccrs.PlateCarree())
        t2m_vmax = np.ceil(max(abs(t2m_diff_map.min()), abs(t2m_diff_map.max())))
        cf3 = ax3.tricontourf(
            LON[plot_region_indexs],
            LAT[plot_region_indexs],
            t2m_diff_map,
            levels=12,
            cmap=ncp.pal("Cobalt"),
            vmin=-t2m_vmax,
            vmax=t2m_vmax,
        )
        contours3 = ax3.tricontour(
            LON[plot_region_indexs],
            LAT[plot_region_indexs],
            z500_hot_comp - z500_cold_comp,
            levels=12,
            colors="black",
        )
        ax3.clabel(contours3, fmt="%.0f", inline=True)
        ax3.coastlines()

        lon_min, lon_max = average_longitude_bnd.start, average_longitude_bnd.stop
        lat_min, lat_max = average_latitude_bnd.start, average_latitude_bnd.stop
        ax3.plot(
            [lon_min, lon_max, lon_max, lon_min, lon_min],
            [lat_min, lat_min, lat_max, lat_max, lat_min],
            color="green",
            linewidth=2,
            transform=ccrs.PlateCarree(),
        )

        ax3.set_title(
            "(c)",  # Surface temperature difference (cutoff - w/ cutoff)",
        )
        cbar3 = fig.colorbar(
            cf3,
            ax=ax3,
            orientation="vertical",
            shrink=0.7,
            ticks=mtickers.MaxNLocator(integer=True),
        )
        cbar3.set_label("Temperature difference ($^\circ$C)")

        # plot vertical profile of temperature difference
        ax4 = fig.add_subplot(2, 2, 4)
        lon_range = np.max(LON[plot_region_indexs]) - np.min(LON[plot_region_indexs])
        lat_range = np.max(LAT[plot_region_indexs]) - np.min(LAT[plot_region_indexs])
        ax4.set_box_aspect(lat_range / lon_range)
        for i, (p_date, prof) in enumerate(profiles.items()):
            day_str = pd.to_datetime(p_date).strftime("%b %d")
            ax4.plot(
                prof,
                levels,
                marker="o",
                #linewidth=2,
                label=f"{day_str}",
            )
        ax4.axvline(0, color="black", linestyle="--", linewidth=1)
        ax4.set_ylabel("Pressure level (hPa)")
        ax4.set_xlabel("Temperature difference ($^\circ$C)")
        ax4.yaxis.tick_right() 
        ax4.yaxis.set_label_position("right")
        ax4.set_title(
            "(d)",  # Vertical profile of temperature differences (averaged over region)",
        )
        ax4.grid(linestyle=":")
        ax4.invert_yaxis()  # invert axis so the ground is on the bottom
        ax4.legend(loc="best", fontsize="x-large")
        ax4.tick_params(axis="both")
        #fake colorbar so it's aligned lol
        cbar4 = fig.colorbar(
            cf3,
            ax=ax4, 
            orientation="vertical", 
            shrink=0.7
        )
        cbar4.ax.set_visible(False)

        plt.tight_layout()
        plt.savefig(
            f"MAY_2026_HEATWAVE/diagnostic_cutoff_no_cutoff_25th_forecast_starting_{k}_{subset_size}_members.pdf",
            bbox_inches="tight",
        )

        plt.show()

        print(
            "-----------------------------------------------------------------------------------------------------------------------------"
        )

## Decomposition of temperature

#### Func

In [ ]:
def temperature_decomposition(v, region, target_member):
    """Perform a vertical and horizontal decomposition of the thermodynamic temperature equation.

    Calculates the components of temperature change (horizontal advection, adiabatic 
    compression, vertical displacement, and diabatic heating residual) at multiple 
    pressure levels over time for a specific spatial region.

    Args:
        v: xarray Dataset containing temperature, u/v/w winds, and geopotential on pressure levels.
        region: Boolean mask or slice indicating the spatial region to analyze.
        target_member: Integer indicating the ensemble member to decompose.

    Returns:
        T, dTdt, adv, Qdiab, comp, disp, vert, valid_times, levels, target_member, lon_min, lon_max, lat_min, lat_max:
        A tuple containing numpy arrays for the spatial averages of temperature, total derivative, 
        advection, diabatic heating, compression, displacement, combined vertical term, along 
        with corresponding temporal and spatial metadata.
    """
    # Target pressure levels
    levels = [1000, 925, 850, 700, 600, 500, 400, 300]
    # pressur levels in pascal
    levels_pa = [lvl * 100 for lvl in levels]

    # load what we'll use so it's faster
    v = v[[f"{var}_{lvl}" for lvl in levels for var in ["t", "u", "v", "w"]]].sel(number=target_member).isel(values=region).load()

    # stack temperature fields cos they're the only ones we'll differentiate vertically
    T_stacked = xr.concat([v[f"t_{lvl}"] for lvl in levels], dim=pd.Index(levels_pa, name="level"))

    # differentiation wrt time and pressure levels
    dT_dp_stacked = T_stacked.differentiate("level")
    dT_dt_stacked = T_stacked.differentiate("step") * 1e9 # step is in timedelta64[ns], differentiating gives K/ns --> multiply by 1e9 for K/s

    # fixed constants/values
    start_step = 0  # 1
    num_steps = 60  # 59
    end_step = start_step + num_steps
    times = np.arange(start_step, end_step)
    SEC_PER_DAY = 86400
    KAPPA = 0.286

    lon_1d = LON[region]
    lat_1d = LAT[region]

    # compute the grid parameters with random data
    grid_lon, grid_lat, _ = interpolate_to_grid(
        lon_1d,
        lat_1d,
        v["t_850"].isel(step=start_step).values,
        interp_type="linear",
        hres=0.25,
    )
    dx, dy = mpcalc.lat_lon_grid_deltas(grid_lon, grid_lat)

    # empty results matrixes
    shape = (len(levels), len(times))
    adv = np.zeros(shape)
    comp = np.zeros(shape)
    disp = np.zeros(shape)
    Qdiab = np.zeros(shape)
    dTdt = np.zeros(shape)
    T = np.zeros(shape)

    # COMPUTATION LOOP
    for t_idx, current_step in enumerate(times):
        v_step = v.isel(step=current_step)
        dT_dp_step = dT_dp_stacked.isel(step=current_step)
        dT_dt_step = dT_dt_stacked.isel(step=current_step)
        
        for l_idx, lvl in enumerate(levels):
            # extract 1D data for current level and step
            T_1d = v_step[f"t_{lvl}"].values
            u_1d = v_step[f"u_{lvl}"].values
            v_1d = v_step[f"v_{lvl}"].values
            w_1d = v_step[f"w_{lvl}"].values
            dT_dp_1d = dT_dp_step.sel(level=lvl * 100).values
            dT_dt_1d = dT_dt_step.sel(level=lvl * 100).values
            
            # interp to 2d for advection (first values returned are the grid so we don't gaf)
            _, _, T_2d = interpolate_to_grid(lon_1d, lat_1d, T_1d, interp_type="linear", hres=0.25)
            _, _, u_2d = interpolate_to_grid(lon_1d, lat_1d, u_1d, interp_type="linear", hres=0.25)
            _, _, v_2d = interpolate_to_grid(lon_1d, lat_1d, v_1d, interp_type="linear", hres=0.25)
            
            # compute advection using metpy
            adv_rate = mpcalc.advection(T_2d * units.kelvin, u=u_2d * units("m/s"), v=v_2d * units("m/s"), dx=dx, dy=dy).m
            
            # compute compression, vertical displascement and Q_diab terms
            comp_rate = w_1d * (KAPPA * T_1d / (lvl * 100))
            disp_rate = -1 * w_1d * dT_dp_1d
            
            # first spatial average so we can compute Q_diab too
            adv_rate_m = np.nanmean(adv_rate)
            comp_rate_m = np.mean(comp_rate)
            disp_rate_m = np.mean(disp_rate)
            dT_dt_1d_m = np.mean(dT_dt_1d)
            
            Qdiab_rate = dT_dt_1d_m - adv_rate_m - comp_rate_m - disp_rate_m
            # not really something we can compute so isolating it
            
            # scale to K/days
            adv[l_idx, t_idx] = adv_rate_m * SEC_PER_DAY
            comp[l_idx, t_idx] = comp_rate_m * SEC_PER_DAY
            disp[l_idx, t_idx] = disp_rate_m * SEC_PER_DAY
            Qdiab[l_idx, t_idx] = Qdiab_rate * SEC_PER_DAY
            dTdt[l_idx, t_idx] = dT_dt_1d_m * SEC_PER_DAY
            T[l_idx, t_idx] = np.mean(T_1d)

    # combined vertical term
    vert = comp + disp

    return T, dTdt, adv, Qdiab, comp, disp, vert, v.valid_time.isel(step=times).values, levels, target_member, np.min(lon_1d), np.max(lon_1d), np.min(lat_1d), np.max(lat_1d)

In [ ]:
def plot_vertical_decomposition_composites(
    v,
    adv1,
    adv2,
    vert1,
    vert2,
    T1,
    T2,
    plot_dates,
    levels,
    temp_in_box1,
    temp_in_box2,
    members1,
    members2,
    save,
    ens1_desc,
    ens2_desc,
    lon_min,
    lon_max,
    lat_min,
    lat_max,
):
    # DATA FOR THE MAP
    plot_lons = LON[plot_region_indexs]
    plot_lats = LAT[plot_region_indexs]
    mask_plot_date = np.isin(
        v.valid_time.values,
        [
            np.datetime64("2026-05-25T00:00"),
            np.datetime64("2026-05-25T06:00"),
            np.datetime64("2026-05-25T12:00"),
            np.datetime64("2026-05-25T18:00"),
        ],  # , np.datetime64("2026-05-26T12:00")],
    )
    z_1d = (
        v["z_500"]
        .isel(step=mask_plot_date)
        .sel(number=members1)
        .isel(values=plot_region_indexs)
        .mean(["step", "number"])
    ).values / 9.81 - (
        v["z_500"]
        .isel(step=mask_plot_date)
        .sel(number=members2)
        .isel(values=plot_region_indexs)
        .mean(["step", "number"])
    ).values / 9.81
    ts_1d = (
        v["2t"]
        .isel(step=mask_plot_date)
        .sel(number=members1)
        .isel(values=plot_region_indexs)
        .mean(["step", "number"])
    ).values - (
        v["2t"]
        .isel(step=mask_plot_date)
        .sel(number=members2)
        .isel(values=plot_region_indexs)
        .mean(["step", "number"])
    ).values

    plot_data = [
        (T1 - T2, "(a)"),  # Temperature"),
        (adv1 - adv2, r"(b)"),  # Horizontal Advection ($-V_h \cdot \nabla_h T$)"),
        (
            vert1 - vert2,
            r"(c)",
        ),  # Adiabatic Heating $\omega \left( \frac{\kappa T}{p}-\frac{\partial T}{\partial p} \right)$",,
    ]

    fig, axes = plt.subplots(
        nrows=2,
        ncols=2,
        figsize=get_figsize(WIDTH_INSA, 1, 0.67),
    )
    axes = axes.flatten()

    for idx, (ax, (data, title)) in enumerate(zip(axes, plot_data)):
        plot_cmap = "RdBu_r"  # ncp.pal("Vanadium")
        plot_levels = np.linspace(-10, 10, 20)

        cf = ax.contourf(
            plot_dates,
            levels,
            data,
            levels=plot_levels,
            cmap=plot_cmap,
            extend="both",
        )

        if title in [r"(b)",r"(c)"]:
            fig.colorbar(
                cf,
                ax=ax,
                pad=0.02,
                label="Heating (K/day)",
                ticks=mtickers.MaxNLocator(integer=True),
                shrink=0.85
            )
        else:
            fig.colorbar(
                cf,
                ax=ax,
                pad=0.02,
                label="Temperature difference ($^\circ$C)",
                ticks=mtickers.MaxNLocator(integer=True),
                shrink=0.85
            )

        ax.set_title(title)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d-%m-%y"))
        ax.tick_params(axis="both", bottom=True, left=True, length=2)
        ax.tick_params(axis="x", rotation=22.5)
        ax.invert_yaxis()
        if title in ["(a)", "(c)"]:
            ax.set_ylabel("Pressure (hPa)")
        if title in ["(a)", r"(b)"]:
            ax.tick_params(labelbottom=False)

    axes[3].remove()
    ax = fig.add_subplot(2, 2, 4, projection=ccrs.PlateCarree())

    ax.coastlines()
    cf_map = ax.tricontourf(
        plot_lons,
        plot_lats,
        ts_1d,
        levels=np.linspace(-9, 9, 18),
        cmap=ncp.pal("Cobalt"),
        transform=ccrs.PlateCarree(),
    )
    contours = ax.tricontour(
        plot_lons,
        plot_lats,
        z_1d,
        levels=12,
        colors="black",
        linewidths=0.7,
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    ax.plot(
        [lon_min, lon_max, lon_max, lon_min, lon_min],
        [lat_min, lat_min, lat_max, lat_max, lat_min],
        color="green",
        linewidth=2,
        transform=ccrs.PlateCarree(),
    )
    ax.clabel(contours, inline=True, fmt="%.0f")

    ax.set_title("(d)")

    fig.colorbar(
        cf_map,
        ax=ax,
        orientation="vertical",
        label="Temperature anomaly ($^\circ$C)",
        pad=0.02,
        shrink=0.85,
        ticks=mtickers.MaxNLocator(integer=True),
    )

    # plt.suptitle(
    #     f"Temperature Decomposition ({len(members1)} ENS1 - {len(members2)} ENS2)\nAverage temperature difference between ENS1 and ENS2={temp_in_box1-temp_in_box2:.1f}$^\circ$C\nSpatially averaged over green box \nENS1:{ens1_desc}\nENS2:{ens2_desc}",
    #     fontsize="large",
    # )

    plt.tight_layout()

    plt.savefig(
        f"hovmollers/forecast_started_{plot_dates[0].astype(str)[:10].replace('-', '')}_composites_{save}.pdf",
        bbox_inches="tight",
    )
    plt.show()

### Composites

#### hot vs cold with threshold

In [ ]:
v = datasets_results["2026051700"].sel(number=members_exceeding["2026051700"])
region = average_region
time_mask = np.isin(
    v.valid_time.values,
    [
        np.datetime64("2026-05-25T12:00"),
    ],
)
# sorting the surface temperatures
t2m_max_per_member = (
    v["2t"].isel(values=region).isel(step=time_mask).mean(dim="values").max(dim="step")
)
t2m_sorted = t2m_max_per_member.sortby(t2m_max_per_member, ascending=False)
members_hot = t2m_sorted.number[0:10].values
members_cold = t2m_sorted.number[-11:-1].values

In [ ]:
adv_hot_list = []
vert_hot_list = []
T_hot_list = []
member_hot_temp_list = []

adv_cold_list = []
vert_cold_list = []
T_cold_list = []
member_cold_temp_list = []

for member_hot in tqdm(members_hot):
    member_hot_temp = t2m_sorted.sel(number=member_hot).values - 273.15

    (
        T_hot,
        dTdt_hot,
        adv_hot,
        Qdiab_hot,
        comp_hot,
        disp_hot,
        vert_hot,
        times,
        levels,
        member_hot,
        lon_min,
        lon_max,
        lat_min,
        lat_max
    ) = temperature_decomposition(
        v=v, region=region, target_member=member_hot
    )

    adv_hot_list.append(adv_hot)
    vert_hot_list.append(vert_hot)
    T_hot_list.append(T_hot)
    member_hot_temp_list.append(member_hot_temp)

for member_cold in tqdm(members_cold):
    member_cold_temp = t2m_sorted.sel(number=member_cold).values - 273.15

    (
        T_cold,
        dTdt_cold,
        adv_cold,
        Qdiab_cold,
        comp_cold,
        disp_cold,
        vert_cold,
        times,
        levels,
        member_cold,
        lon_min,
        lon_max,
        lat_min,
        lat_max
    ) = temperature_decomposition(
        v=v, region=region, target_member=member_cold
    )

    adv_cold_list.append(adv_cold)
    vert_cold_list.append(vert_cold)
    T_cold_list.append(T_cold)
    member_cold_temp_list.append(member_cold_temp)

adv_hot_stacked = np.stack(adv_hot_list)
vert_hot_stacked = np.stack(vert_hot_list)
T_hot_stacked = np.stack(T_hot_list)
member_hot_temp_stacked = np.stack(member_hot_temp_list)
adv_cold_stacked = np.stack(adv_cold_list)
vert_cold_stacked = np.stack(vert_cold_list)
T_cold_stacked = np.stack(T_cold_list)
member_cold_temp_stacked = np.stack(member_cold_temp_list)

adv_hot_mean = np.mean(adv_hot_stacked, axis=0)
vert_hot_mean = np.mean(vert_hot_stacked, axis=0)
T_hot_mean = np.mean(T_hot_stacked, axis=0)
member_hot_temp_mean = np.mean(member_hot_temp_stacked, axis=0)
adv_cold_mean = np.mean(adv_cold_stacked, axis=0)
vert_cold_mean = np.mean(vert_cold_stacked, axis=0)
T_cold_mean = np.mean(T_cold_stacked, axis=0)
member_cold_temp_mean = np.mean(member_cold_temp_stacked, axis=0)

In [ ]:
plot_vertical_decomposition_composites(
    v,
    adv_hot_mean,
    adv_cold_mean,
    vert_hot_mean,
    vert_cold_mean,
    T_hot_mean,
    T_cold_mean,
    times,
    levels,
    member_hot_temp_mean,
    member_cold_temp_mean,
    members_hot,
    members_cold,
    "hot_minus_cold_10best_each_with_threshold_target_region",
    "Hottest members",
    "Coldest members",
    lon_min,
    lon_max,
    lat_min,
    lat_max
)

In [ ]:
v = datasets_results["2026051800"].sel(number=members_exceeding["2026051800"])
region = average_region
time_mask = np.isin(
    v.valid_time.values,
    [
        np.datetime64("2026-05-25T12:00"),
    ],
)
# sorting the surface temperatures
t2m_max_per_member = (
    v["2t"].isel(values=region).isel(step=time_mask).mean(dim="values").max(dim="step")
)
t2m_sorted = t2m_max_per_member.sortby(t2m_max_per_member, ascending=False)
members_hot = t2m_sorted.number[0:10].values
members_cold = t2m_sorted.number[-11:-1].values

In [ ]:
adv_hot_list = []
vert_hot_list = []
T_hot_list = []
member_hot_temp_list = []

adv_cold_list = []
vert_cold_list = []
T_cold_list = []
member_cold_temp_list = []

for member_hot in tqdm(members_hot):
    member_hot_temp = t2m_sorted.sel(number=member_hot).values - 273.15

    (
        T_hot,
        dTdt_hot,
        adv_hot,
        Qdiab_hot,
        comp_hot,
        disp_hot,
        vert_hot,
        times,
        levels,
        member_hot,
        lon_min,
        lon_max,
        lat_min,
        lat_max
    ) = temperature_decomposition(
        v=v, region=region, target_member=member_hot
    )

    adv_hot_list.append(adv_hot)
    vert_hot_list.append(vert_hot)
    T_hot_list.append(T_hot)
    member_hot_temp_list.append(member_hot_temp)

for member_cold in tqdm(members_cold):
    member_cold_temp = t2m_sorted.sel(number=member_cold).values - 273.15

    (
        T_cold,
        dTdt_cold,
        adv_cold,
        Qdiab_cold,
        comp_cold,
        disp_cold,
        vert_cold,
        times,
        levels,
        member_cold,
        lon_min,
        lon_max,
        lat_min,
        lat_max
    ) = temperature_decomposition(
        v=v, region=region, target_member=member_cold
    )

    adv_cold_list.append(adv_cold)
    vert_cold_list.append(vert_cold)
    T_cold_list.append(T_cold)
    member_cold_temp_list.append(member_cold_temp)

adv_hot_stacked = np.stack(adv_hot_list)
vert_hot_stacked = np.stack(vert_hot_list)
T_hot_stacked = np.stack(T_hot_list)
member_hot_temp_stacked = np.stack(member_hot_temp_list)
adv_cold_stacked = np.stack(adv_cold_list)
vert_cold_stacked = np.stack(vert_cold_list)
T_cold_stacked = np.stack(T_cold_list)
member_cold_temp_stacked = np.stack(member_cold_temp_list)

adv_hot_mean = np.mean(adv_hot_stacked, axis=0)
vert_hot_mean = np.mean(vert_hot_stacked, axis=0)
T_hot_mean = np.mean(T_hot_stacked, axis=0)
member_hot_temp_mean = np.mean(member_hot_temp_stacked, axis=0)
adv_cold_mean = np.mean(adv_cold_stacked, axis=0)
vert_cold_mean = np.mean(vert_cold_stacked, axis=0)
T_cold_mean = np.mean(T_cold_stacked, axis=0)
member_cold_temp_mean = np.mean(member_cold_temp_stacked, axis=0)

In [ ]:
plot_vertical_decomposition_composites(
    v,
    adv_hot_mean,
    adv_cold_mean,
    vert_hot_mean,
    vert_cold_mean,
    T_hot_mean,
    T_cold_mean,
    times,
    levels,
    member_hot_temp_mean,
    member_cold_temp_mean,
    members_hot,
    members_cold,
    "hot_minus_cold_10best_each_with_threshold_target_region",
    "Hottest members",
    "Coldest members",
    lon_min,
    lon_max,
    lat_min,
    lat_max
)

#### With low pressure area vs without

In [ ]:
v = datasets_results["2026051700"].sel(number=members_exceeding["2026051700"])
region = average_region
time_mask = np.isin(
    v.valid_time.values,
    [
        np.datetime64("2026-05-25T12:00"),
    ],
)
# sorting the surface temperatures
t2m_max_per_member = (
    v["2t"].isel(values=region).isel(step=time_mask).mean(dim="values").max(dim="step")
)

In [ ]:
adv_with_low_pressure_area_list = []
vert_with_low_pressure_area_list = []
T_with_low_pressure_area_list = []
member_with_low_pressure_area_temp_list = []

adv_without_low_pressure_area_list = []
vert_without_low_pressure_area_list = []
T_without_low_pressure_area_list = []
member_without_low_pressure_area_temp_list = []

for member_with_low_pressure_area in tqdm(members_with_low_pressure_area_17th_top10):
    member_with_low_pressure_area_temp = t2m_max_per_member.sel(number=member_with_low_pressure_area).values - 273.15

    (
        T_with_low_pressure_area,
        dTdt_with_low_pressure_area,
        adv_with_low_pressure_area,
        Qdiab_with_low_pressure_area,
        comp_with_low_pressure_area,
        disp_with_low_pressure_area,
        vert_with_low_pressure_area,
        times,
        levels,
        member_with_low_pressure_area,
        lon_min,
        lon_max,
        lat_min,
        lat_max
    ) = temperature_decomposition(
        v=v, region=region, target_member=member_with_low_pressure_area
    )

    adv_with_low_pressure_area_list.append(adv_with_low_pressure_area)
    vert_with_low_pressure_area_list.append(vert_with_low_pressure_area)
    T_with_low_pressure_area_list.append(T_with_low_pressure_area)
    member_with_low_pressure_area_temp_list.append(member_with_low_pressure_area_temp)
#len(members_with_low_pressure_area)
for member_without_low_pressure_area in tqdm(members_without_low_pressure_area_17th):
    member_without_low_pressure_area_temp = t2m_max_per_member.sel(number=member_without_low_pressure_area).values - 273.15

    (
        T_without_low_pressure_area,
        dTdt_without_low_pressure_area,
        adv_without_low_pressure_area,
        Qdiab_without_low_pressure_area,
        comp_without_low_pressure_area,
        disp_without_low_pressure_area,
        vert_without_low_pressure_area,
        times,
        levels,
        member_without_low_pressure_area,
        lon_min,
        lon_max,
        lat_min,
        lat_max
    ) = temperature_decomposition(
        v=v, region=region, target_member=member_without_low_pressure_area
    )

    adv_without_low_pressure_area_list.append(adv_without_low_pressure_area)
    vert_without_low_pressure_area_list.append(vert_without_low_pressure_area)
    T_without_low_pressure_area_list.append(T_without_low_pressure_area)
    member_without_low_pressure_area_temp_list.append(member_without_low_pressure_area_temp)

adv_with_low_pressure_area_stacked = np.stack(adv_with_low_pressure_area_list)
vert_with_low_pressure_area_stacked = np.stack(vert_with_low_pressure_area_list)
T_with_low_pressure_area_stacked = np.stack(T_with_low_pressure_area_list)
member_with_low_pressure_area_temp_stacked = np.stack(member_with_low_pressure_area_temp_list)
adv_without_low_pressure_area_stacked = np.stack(adv_without_low_pressure_area_list)
vert_without_low_pressure_area_stacked = np.stack(vert_without_low_pressure_area_list)
T_without_low_pressure_area_stacked = np.stack(T_without_low_pressure_area_list)
member_without_low_pressure_area_temp_stacked = np.stack(member_without_low_pressure_area_temp_list)

adv_with_low_pressure_area_mean = np.mean(adv_with_low_pressure_area_stacked, axis=0)
vert_with_low_pressure_area_mean = np.mean(vert_with_low_pressure_area_stacked, axis=0)
T_with_low_pressure_area_mean = np.mean(T_with_low_pressure_area_stacked, axis=0)
member_with_low_pressure_area_temp_mean = np.mean(member_with_low_pressure_area_temp_stacked, axis=0)
adv_without_low_pressure_area_mean = np.mean(adv_without_low_pressure_area_stacked, axis=0)
vert_without_low_pressure_area_mean = np.mean(vert_without_low_pressure_area_stacked, axis=0)
T_without_low_pressure_area_mean = np.mean(T_without_low_pressure_area_stacked, axis=0)
member_without_low_pressure_area_temp_mean = np.mean(member_without_low_pressure_area_temp_stacked, axis=0)

In [ ]:
plot_vertical_decomposition_composites(
    v,
    adv_with_low_pressure_area_mean,
    adv_without_low_pressure_area_mean,
    vert_with_low_pressure_area_mean,
    vert_without_low_pressure_area_mean,
    T_with_low_pressure_area_mean,
    T_without_low_pressure_area_mean,
    times,
    levels,
    member_with_low_pressure_area_temp_mean,
    member_without_low_pressure_area_temp_mean,
    members_with_low_pressure_area_17th_top10,
    members_without_low_pressure_area_17th,
    "with_low_pressure_area_minus_without_low_pressure_area_10best_each_with_threshold_target_region",
    "Members with cutoff low or deep trough",
    "Members without",
    lon_min,
    lon_max,
    lat_min,
    lat_max
)

In [ ]:
v = datasets_results["2026051800"].sel(number=members_exceeding["2026051800"])
region = average_region
time_mask = np.isin(
    v.valid_time.values,
    [
        np.datetime64("2026-05-25T12:00"),
    ],
)
# sorting the surface temperatures
t2m_max_per_member = (
    v["2t"].isel(values=region).isel(step=time_mask).mean(dim="values").max(dim="step")
)

In [ ]:
adv_with_low_pressure_area_list = []
vert_with_low_pressure_area_list = []
T_with_low_pressure_area_list = []
member_with_low_pressure_area_temp_list = []

adv_without_low_pressure_area_list = []
vert_without_low_pressure_area_list = []
T_without_low_pressure_area_list = []
member_without_low_pressure_area_temp_list = []

for member_with_low_pressure_area in tqdm(members_with_low_pressure_area_18th_top10):
    member_with_low_pressure_area_temp = t2m_max_per_member.sel(number=member_with_low_pressure_area).values - 273.15

    (
        T_with_low_pressure_area,
        dTdt_with_low_pressure_area,
        adv_with_low_pressure_area,
        Qdiab_with_low_pressure_area,
        comp_with_low_pressure_area,
        disp_with_low_pressure_area,
        vert_with_low_pressure_area,
        times,
        levels,
        member_with_low_pressure_area,
        lon_min,
        lon_max,
        lat_min,
        lat_max
    ) = temperature_decomposition(
        v=v, region=region, target_member=member_with_low_pressure_area
    )

    adv_with_low_pressure_area_list.append(adv_with_low_pressure_area)
    vert_with_low_pressure_area_list.append(vert_with_low_pressure_area)
    T_with_low_pressure_area_list.append(T_with_low_pressure_area)
    member_with_low_pressure_area_temp_list.append(member_with_low_pressure_area_temp)
#len(members_with_low_pressure_area)
for member_without_low_pressure_area in tqdm(members_without_low_pressure_area_18th):
    member_without_low_pressure_area_temp = t2m_max_per_member.sel(number=member_without_low_pressure_area).values - 273.15

    (
        T_without_low_pressure_area,
        dTdt_without_low_pressure_area,
        adv_without_low_pressure_area,
        Qdiab_without_low_pressure_area,
        comp_without_low_pressure_area,
        disp_without_low_pressure_area,
        vert_without_low_pressure_area,
        times,
        levels,
        member_without_low_pressure_area,
        lon_min,
        lon_max,
        lat_min,
        lat_max
    ) = temperature_decomposition(
        v=v, region=region, target_member=member_without_low_pressure_area
    )

    adv_without_low_pressure_area_list.append(adv_without_low_pressure_area)
    vert_without_low_pressure_area_list.append(vert_without_low_pressure_area)
    T_without_low_pressure_area_list.append(T_without_low_pressure_area)
    member_without_low_pressure_area_temp_list.append(member_without_low_pressure_area_temp)

adv_with_low_pressure_area_stacked = np.stack(adv_with_low_pressure_area_list)
vert_with_low_pressure_area_stacked = np.stack(vert_with_low_pressure_area_list)
T_with_low_pressure_area_stacked = np.stack(T_with_low_pressure_area_list)
member_with_low_pressure_area_temp_stacked = np.stack(member_with_low_pressure_area_temp_list)
adv_without_low_pressure_area_stacked = np.stack(adv_without_low_pressure_area_list)
vert_without_low_pressure_area_stacked = np.stack(vert_without_low_pressure_area_list)
T_without_low_pressure_area_stacked = np.stack(T_without_low_pressure_area_list)
member_without_low_pressure_area_temp_stacked = np.stack(member_without_low_pressure_area_temp_list)

adv_with_low_pressure_area_mean = np.mean(adv_with_low_pressure_area_stacked, axis=0)
vert_with_low_pressure_area_mean = np.mean(vert_with_low_pressure_area_stacked, axis=0)
T_with_low_pressure_area_mean = np.mean(T_with_low_pressure_area_stacked, axis=0)
member_with_low_pressure_area_temp_mean = np.mean(member_with_low_pressure_area_temp_stacked, axis=0)
adv_without_low_pressure_area_mean = np.mean(adv_without_low_pressure_area_stacked, axis=0)
vert_without_low_pressure_area_mean = np.mean(vert_without_low_pressure_area_stacked, axis=0)
T_without_low_pressure_area_mean = np.mean(T_without_low_pressure_area_stacked, axis=0)
member_without_low_pressure_area_temp_mean = np.mean(member_without_low_pressure_area_temp_stacked, axis=0)

In [ ]:
plot_vertical_decomposition_composites(
    v,
    adv_with_low_pressure_area_mean,
    adv_without_low_pressure_area_mean,
    vert_with_low_pressure_area_mean,
    vert_without_low_pressure_area_mean,
    T_with_low_pressure_area_mean,
    T_without_low_pressure_area_mean,
    times,
    levels,
    member_with_low_pressure_area_temp_mean,
    member_without_low_pressure_area_temp_mean,
    members_with_low_pressure_area_18th_top10,
    members_without_low_pressure_area_18th,
    "with_low_pressure_area_minus_without_low_pressure_area_10best_each_with_threshold_target_region",
    "Members with cutoff low or deep trough",
    "Members without",
    lon_min,
    lon_max,
    lat_min,
    lat_max
)

## Return times

#### era5 full data at 12utc in paris for may

In [ ]:
era5_paris_12utc_may=era5_paris_ts.isel(valid_time=era5_paris_ts.valid_time.dt.month.isin([5])*era5_paris_ts.valid_time.dt.hour.isin([12])).t2m-273.15
era5_paris_12utc_may_max_per_year=era5_paris_12utc_may.groupby("valid_time.year").max()

In [ ]:
era5_london_12utc_may=era5_london_ts.isel(valid_time=era5_london_ts.valid_time.dt.month.isin([5])*era5_london_ts.valid_time.dt.hour.isin([12])).t2m-273.15
era5_london_12utc_may_max_per_year=era5_london_12utc_may.groupby("valid_time.year").max()

#### RMST

In [ ]:
europe_temp = (
    xr.open_dataset("gis_europe_temp.nc.gz")
    .rename_vars({"tempanomaly": "t2m"})
    .t2m
)
europe_temp = (
    europe_temp.sel(lat=slice(30, 70), lon=slice(-20, 50))
    .mean(["lat", "lon"])
    .groupby("time.year")
    .mean()
    .rolling(year=15, center=True)
    .mean()
    .interpolate_na(dim="year", method="linear", fill_value="extrapolate")
)
europe_temp_per_year=europe_temp.sel(year=europe_temp.year >= 1940)

#### detrending

In [ ]:
slope_europe, _ = np.polyfit(europe_temp_per_year, era5_paris_12utc_may_max_per_year, deg=1)
rmst_2026_europe = europe_temp_per_year.isel(year=-1)

In [ ]:
slope_europe_london, _ = np.polyfit(europe_temp_per_year, era5_london_12utc_may_max_per_year, deg=1)

In [ ]:
era5_paris_12utc_may_max_per_year_detrended_europe = era5_paris_12utc_may_max_per_year + slope_europe * (rmst_2026_europe - europe_temp_per_year)
era5_paris_12utc_may_detrended_europe = era5_paris_12utc_may + slope_europe * (rmst_2026_europe - europe_temp_per_year.sel(year=era5_paris_12utc_may.valid_time.dt.year))

In [ ]:
era5_london_12utc_may_max_per_year_detrended_europe = era5_london_12utc_may_max_per_year + slope_europe * (rmst_2026_europe - europe_temp_per_year)
era5_london_12utc_may_detrended_europe = era5_london_12utc_may + slope_europe * (rmst_2026_europe - europe_temp_per_year.sel(year=era5_london_12utc_may.valid_time.dt.year))

#### Finding the extremes

In [ ]:
detrended_maxes = era5_paris_12utc_may_detrended_europe.groupby("valid_time.year").max()
peak_dates = [
    year_data.idxmax(dim="valid_time").values 
    for year, year_data in era5_paris_12utc_may_detrended_europe.groupby("valid_time.year")
]
df_peaks = pd.DataFrame({
    "Detrended_Max_Temp_C": detrended_maxes.values,
    "Exact_Date": peak_dates
}, index=detrended_maxes.year.values)
df_peaks.index.name = "Year"
df_peaks_sorted = df_peaks.sort_values(by="Detrended_Max_Temp_C", ascending=False)

print(df_peaks_sorted.head(10))

In [ ]:
detrended_maxes = era5_london_12utc_may_detrended_europe.groupby("valid_time.year").max()
peak_dates = [
    year_data.idxmax(dim="valid_time").values 
    for year, year_data in era5_london_12utc_may_detrended_europe.groupby("valid_time.year")
]
df_peaks = pd.DataFrame({
    "Detrended_Max_Temp_C": detrended_maxes.values,
    "Exact_Date": peak_dates
}, index=detrended_maxes.year.values)
df_peaks.index.name = "Year"
df_peaks_sorted = df_peaks.sort_values(by="Detrended_Max_Temp_C", ascending=False)

print(df_peaks_sorted.head(10))

#### func

In [ ]:
# (Bloin-Wibe et al, 2025)

def get_boosted_return_periods_and_gev(era5, boosted_maxes,N_boot=1000,N_parent=1,T_ref=None):
    """Calculate boosted return periods and confidence intervals.

    Implements the Bloin-Wibe et al. (2025) boosted estimator to evaluate extreme 
    return periods by combining historical observations with ensemble forecasts.

    Args:
        era5: xarray DataArray of historical ERA5 data.
        boosted_maxes: Array of maximum values extracted from the boosted ensemble data.
        N_boot: Number of bootstrap iterations for confidence intervals. Defaults to 1000.
        N_parent: Effective size parameter for the reference probability. Defaults to 1.
        T_ref: Reference temperature threshold for conditioning.

    Returns:
        return_periods, T_ext_array, ci_lower, ci_upper: A tuple containing the computed
        return periods, the extrapolated temperature array, and the lower/upper bounds 
        of the 95% confidence interval.
    """
    
    # yearly climato for may
    era5_yearly_max = era5.groupby('valid_time.year').max(dim='valid_time')
    
    # boosted data
    N_boosted_total = len(boosted_maxes)
    T_ext_max = boosted_maxes.max()
    # P(T >= T_ref)
    N_years = len(era5_yearly_max)
    P_T_ref = N_parent / (N_years + 1) 

    #return periods
    T_ext_array = np.linspace(T_ref, T_ext_max, 100)
    return_periods = get_return_periods(boosted_maxes, N_boosted_total, T_ref, P_T_ref, T_ext_array)

    #confint
    ci_lower, ci_upper = get_boosted_bootstrap_ci(boosted_maxes, N_boosted_total, T_ref, P_T_ref, T_ext_array, return_periods, N=N_boot)

    return return_periods, T_ext_array, ci_lower, ci_upper
    
def get_gev(ts):
    # GEV
    model = EVA(data=ts)
    model.get_extremes(
        method="BM",
        extremes_type="high",
        block_size="365.2425D"
    )
    model.fit_model(model="Emcee")

    return model

def get_return_periods(boosted_maxes, N_boosted_total, T_ref, P_T_ref, T_ext_array):
    """Calculate unconditional return periods for a range of extreme temperature thresholds.

    Applies (Bloin-Wide et Al, 2025)

    Args:
        boosted_maxes: Array of maximum values extracted from the boosted ensemble data.
        N_boosted_total: Total number of members/samples in the boosted dataset.
        T_ref: Reference temperature threshold for conditioning.
        P_T_ref: Probability of exceeding the reference temperature in the parent distribution.
        T_ext_array: Array of extreme temperature thresholds to evaluate.

    Returns:
        return_periods: List of calculated return periods corresponding to T_ext_array.
    """
    # P(T_ref|AC)
    P_T_ref_AC = np.sum(boosted_maxes >= T_ref) / N_boosted_total
    
    return_periods = []
    
    for T_ext in T_ext_array:
        # P(T_ext|AC)
        P_T_ext_AC = np.sum(boosted_maxes >= T_ext) / N_boosted_total    
        # P_T_ext_uncond = P_T_ref * ( P(T_ext|AC) / P(T_ref|AC) )
        P_T_ext_uncond = P_T_ref * (P_T_ext_AC / P_T_ref_AC)
        #return period
        rp = 1.0 / P_T_ext_uncond
        return_periods.append(rp)
        
    return return_periods

def get_boosted_bootstrap_ci(boosted_maxes, N_boosted_total, T_ref, P_T_ref, T_ext_array, return_periods, N=1000):
    """Compute the 95% bootstrap confidence intervals for boosted return periods.

    Resamples the boosted ensemble maximums N times with replacement to establish 
    upper and lower bounds for the estimated return periods.

    Args:
        boosted_maxes: Array of maximum values extracted from the boosted ensemble data.
        N_boosted_total: Total number of members/samples in the boosted dataset.
        T_ref: Reference temperature threshold.
        P_T_ref: Probability of exceeding the reference temperature in the parent distribution.
        T_ext_array: Array of extreme temperature thresholds evaluated.
        return_periods: List of original calculated return periods.
        N: Number of bootstrap resampling iterations. Defaults to 1000.

    Returns:
        ci_rp_lower, ci_rp_upper: Arrays representing the lower and upper bounds of 
                                  the 95% confidence intervals for the return periods.
    """
    bootstrap_probs = np.full((N, len(T_ext_array)),np.nan)
    
    for i in range(N):
        sample = np.random.choice(boosted_maxes, size=N_boosted_total, replace=True)
        prob_ref_sample = np.sum(sample >= T_ref) / N_boosted_total
        for j, T_ext in enumerate(T_ext_array):
            prob_ext_sample = np.sum(sample >= T_ext) / N_boosted_total
            if prob_ext_sample > 0 and prob_ref_sample > 0:
                bootstrap_probs[i, j] = P_T_ref * (prob_ext_sample / prob_ref_sample)
    
    prob_lower_bound = np.nanpercentile(bootstrap_probs, 2.5, axis=0)
    prob_upper_bound = np.nanpercentile(bootstrap_probs, 97.5, axis=0)
    
    ci_rp_lower = 1.0 / prob_upper_bound
    ci_rp_upper = 1.0 / prob_lower_bound

    ci_rp_lower, ci_rp_upper = np.minimum(ci_rp_lower, ci_rp_upper), np.maximum(ci_rp_lower, ci_rp_upper)

    return ci_rp_lower, ci_rp_upper

def plot_return_times(model, return_periods, T_ext_array, ci_lower, ci_upper, name, hline):
    fig, ax = plt.subplots(figsize=get_figsize(WIDTH_INSA,1,0.6))
    
    # plot historical return times + gev + confint
    obs_temp = np.sort(model.extremes.values)
    obs_rp = (len(obs_temp) + 1.0) / np.arange(len(obs_temp), 0, -1)
    gev_rp = np.logspace(0, 4.7, 100)
    gev_temp, gev_lower, gev_upper = model.get_return_value(gev_rp, alpha=0.95)
    
    ax.scatter(obs_rp, obs_temp, color="black", marker="o", label="Historical Data (ERA5)", alpha= 0.75)
    ax.plot(gev_rp, gev_temp, color="#F85C50", label="GEV Fit on historical data")
    ax.fill_between(gev_rp, gev_lower, gev_upper, color="#5199FF", alpha=0.25, label="95% CI (GEV)")
    # plot boosted estimated return times
    ax.plot(return_periods, T_ext_array, color="black",label='Boosted Estimator using (Bloin-Wibe et al, 2025) + AIFS-Ens')
    # plot their confint 
    ax.fill_betweenx(T_ext_array, ci_lower, ci_upper, color="green", alpha=0.3,
                     label='95% bootstrap CI (AIFS Boosted)')
    ax.axhline(y=hline, ls="--", label="ERA5 May 12UTC Max (2026)")
    
    # nicer plot
    ax.set_xscale("log")
    ax.set_xlim(1,1e4)
    ax.set_ylim(16,40)
    ax.set_xlabel('Estimated return period (Years)')
    ax.set_ylabel('max May 12 UTC temperature ($^\circ$C)')
    ax.grid(True, which="major", ls="-", alpha=0.6)
    ax.grid(True, which="minor", ls=":", alpha=0.4)
    ax.set_yticks(np.arange(16, 42, 2))
    ax.set_yticks(np.arange(16, 41, 1), minor=True)
    ax.legend(loc='lower right')
    plt.tight_layout()
    
    plt.savefig(f"return_times/return_periods_{name}.pdf",bbox_inches="tight")
    plt.show()

In [ ]:
dates = [
    "1944051600",
    "1944051700",
    "1944051800",
    "1947051800",
    "1947051900",
    "1947052000",
    "1953051200",
    "1953051300",
    "1953051400",
    "1969050100",
    "1969050200",
    "1969050300",
    "1976042500",
    "1976042600",
    "1976042700",
    "1992050100",
    "1992050200",
    "1992050300",
    "1998050100",
    "1998050200",
    "1998050300",
    "2005051400",
    "2005051500",
    "2005051600",
    "2010050900",
    "2010051000",
    "2010051100"
]
dates2 = [
    "2026051800",
    "2026051900",
    "2026052000"
]
variables = [
    "2t",
]

base_dir = "/scratchx/pchevali/RETURN_LEVELS_MAY_PROCESSED"
base_dir2 = "/scratchx/pchevali/MAY_2026_HEATWAVE_PROCESSED"

dataset_return_times = {}

for date in dates:
    dataset_return_times[date] = {var: {} for var in variables}

    for var in variables:
        pattern = f"{base_dir}/{date}-*/*-{var}.nc"
        file_paths = glob.glob(pattern)

        ds = xr.open_dataset(
            file_paths[0], decode_timedelta=True, chunks="auto", engine="h5netcdf"
        )
        ds = ds.assign_coords(longitude=(((ds.longitude + 180) % 360) - 180))
        dataset_return_times[date][var] = ds[var]

for date in dates2:
    dataset_return_times[date] = {var: {} for var in variables}

    for var in variables:
        pattern = f"{base_dir2}/{date}-*/*-{var}.nc"
        file_paths = glob.glob(pattern)

        ds = xr.open_dataset(
            file_paths[0], decode_timedelta=True, chunks="auto", engine="h5netcdf"
        )
        ds = ds.assign_coords(longitude=(((ds.longitude + 180) % 360) - 180))
        dataset_return_times[date][var] = ds[var]

for date in dates+dates2:
    dataset_return_times[date] = xr.merge(
        dataset_return_times[date].values(), compat="override"
    )

#### Return periods with europe detrending

##### With all the data

In [ ]:
GEV = get_gev(era5_paris_12utc_may_detrended_europe.squeeze().to_series())

In [ ]:
all_boosted_list = []

for start_date, start_day_data in dataset_return_times.items():
    start_year = int(start_date[:4]) 

    boosted_data = start_day_data["2t"].isel(values=paris_index).isel(
        step=start_day_data.valid_time.dt.hour.isin([12]).compute()
    )-273.15
    
    boosted_data = boosted_data + slope_europe * (rmst_2026_europe - europe_temp_per_year.sel(year=start_year)).item()
    
    boosted_slice = boosted_data#.isel(step=range(0, 18))
    all_boosted_list.append(boosted_slice)

combined_boosted_data = xr.concat(all_boosted_list, dim='batch').max(dim='step').compute().values.flatten()

rp, T_arr, low, up = get_boosted_return_periods_and_gev(era5_paris_12utc_may_detrended_europe, combined_boosted_data, N_boot=1000,T_ref=29.538267, N_parent=10)

In [ ]:
plot_return_times(GEV, rp, T_arr, low, up, name="may_yearly_max_full_data_europe_detrended",hline=31.725372)

In [ ]:
gev_rp_array = np.logspace(0, 3, 1000) 
gev_temp_array, _, _ = GEV.get_return_value(gev_rp_array, alpha=0.95)
gev_rp_2026 = np.interp(31.725372, gev_temp_array, gev_rp_array)

idx_closest_2026 = np.abs(T_arr - 31.725372).argmin()
boosted_rp_2026 = rp[idx_closest_2026]

print(f"Temps de retour estimé (AIFS Boosté) pour la vague de chaleur de mai 2026 : {boosted_rp_2026:.0f} ans")
print(f"Temps de retour estimé (GEV Historique) pour la vague de chaleur de mai 2026 : {gev_rp_2026:.0f} ans")

##### With only <2026

In [ ]:
GEV = get_gev(era5_paris_12utc_may_detrended_europe.squeeze().to_series()[era5_paris_12utc_may_detrended_europe.squeeze().to_series().index<np.datetime64("2026")])

In [ ]:
all_boosted_list = []

for start_date, start_day_data in dataset_return_times.items():
    start_year = int(start_date[:4])
    if start_year >= 2026:
        continue

    boosted_data = start_day_data["2t"].isel(values=paris_index).isel(
        step=start_day_data.valid_time.dt.hour.isin([12]).compute()
    )-273.15
    
    boosted_data = boosted_data + slope_europe * (rmst_2026_europe - europe_temp_per_year.sel(year=start_year)).item()
    
    boosted_slice = boosted_data#.isel(step=range(0, 18))
    all_boosted_list.append(boosted_slice)

combined_boosted_data = xr.concat(all_boosted_list, dim='batch').max(dim='step').compute().values.flatten()

rp, T_arr, low, up = get_boosted_return_periods_and_gev(era5_paris_12utc_may_detrended_europe, combined_boosted_data, N_boot=1000,T_ref=29.538267, N_parent=9)

In [ ]:
plot_return_times(GEV, rp, T_arr, low, up, name="may_yearly_max_no_2026_europe_detrended",hline=31.725372)

In [ ]:
gev_rp_array = np.logspace(0, 3, 1000) 
gev_temp_array, _, _ = GEV.get_return_value(gev_rp_array, alpha=0.95)
gev_rp_2026 = np.interp(31.725372, gev_temp_array, gev_rp_array)

idx_closest_2026 = np.abs(T_arr - 31.725372).argmin()
boosted_rp_2026 = rp[idx_closest_2026]

print(f"Temps de retour estimé (AIFS Boosté) pour la vague de chaleur de mai 2026 : {boosted_rp_2026:.0f} ans")
print(f"Temps de retour estimé (GEV Historique) pour la vague de chaleur de mai 2026 : {gev_rp_2026:.0f} ans")

In [ ]:
GEV

In [ ]:
params = GEV.model.fit_parameters
upper_bound = params['loc'] + (params['scale'] / params['c'])
print(f"Theoretical upper limit: {upper_bound:.3f}°C")

##### For london <2026

In [ ]:
london_latitude = 51.5074
london_longitude = -0.1278
london_index = get_nearest_point(
    datasets_results["2026051500"], london_latitude, london_longitude
)

london_overlap_years = ["1944", "1947", "1953", "2005"]
all_boosted_list_lon = []

for start_date, start_day_data in dataset_return_times.items():
    start_year = start_date[:4]

    if start_year in london_overlap_years:
        boosted_data_lon = (
            start_day_data["2t"]
            .isel(values=london_index)
            .isel(step=start_day_data.valid_time.dt.hour.isin([12]).compute())
            - 273.15
        )

        boosted_data_lon = (
            boosted_data_lon
            + slope_europe_london
            * (rmst_2026_europe - europe_temp_per_year.sel(year=int(start_year))).item()
        )

        all_boosted_list_lon.append(boosted_data_lon)

combined_boosted_data_lon = (
    xr.concat(all_boosted_list_lon, dim="batch")
    .max(dim="step")
    .compute()
    .values.flatten()
)

london_era5_series = era5_london_12utc_may_detrended_europe.squeeze().to_series()
GEV_lon = get_gev(london_era5_series[london_era5_series.index < np.datetime64("2026")])

rp_lon, T_arr_lon, low_lon, up_lon = get_boosted_return_periods_and_gev(
    era5_london_12utc_may_detrended_europe,
    combined_boosted_data_lon,
    N_boot=1000,
    T_ref=28.368356,
    N_parent=6,
)

plot_return_times(
    GEV_lon,
    rp_lon,
    T_arr_lon,
    low_lon,
    up_lon,
    name="may_yearly_max_london_overlap_europe_detrended_no_2026",
    hline=31.065277,  # 2026 exact max for London
)

## Worst case event

In [ ]:
maxs_forecasts15th=datasets_results['2026051500']['2t'].isel(step=range(0,76)).isel(values=paris_index).max("step").values
maxs_forecasts16th=datasets_results['2026051600']['2t'].isel(step=range(0,76)).isel(values=paris_index).max("step").values
maxs_forecasts17th=datasets_results['2026051700']['2t'].isel(step=range(0,76)).isel(values=paris_index).max("step").values
maxs_forecasts18th=datasets_results['2026051800']['2t'].isel(step=range(0,76)).isel(values=paris_index).max("step").values
maxs_forecasts19th=datasets_results['2026051900']['2t'].isel(step=range(0,76)).isel(values=paris_index).max("step").values
maxs_forecasts20th=datasets_results['2026052000']['2t'].isel(step=range(0,76)).isel(values=paris_index).max("step").values
maxs_forecasts21th=datasets_results['2026052100']['2t'].isel(step=range(0,76)).isel(values=paris_index).max("step").values
maxs_forecasts22th=datasets_results['2026052200']['2t'].isel(step=range(0,76)).isel(values=paris_index).max("step").values
maxs_forecasts23th=datasets_results['2026052300']['2t'].isel(step=range(0,76)).isel(values=paris_index).max("step").values
max_era5_2026=era5_paris_ts["t2m"].sel(valid_time="2026-05")
max_era5_2026=max_era5_2026.isel(valid_time=max_era5_2026.valid_time.dt.hour.isin([12])).max().values
print("On the gridpoint closest to paris")
print(f"Amount of members reaching ERA5's May max d'ERA5 ({max_era5_2026-273.15:.2f}$^\circ$C) for a forecast started on (15/05/2026) : {np.sum(maxs_forecasts15th>max_era5_2026)}/{len(maxs_forecasts15th)} ({np.mean(maxs_forecasts15th>max_era5_2026)*100:.2f}%) (max={np.max(maxs_forecasts15th)-273.15:.2f}$^\circ$C)")
print(f"Amount of members reaching ERA5's May max d'ERA5 ({max_era5_2026-273.15:.2f}$^\circ$C) for a forecast started on (16/05/2026) : {np.sum(maxs_forecasts16th>max_era5_2026)}/{len(maxs_forecasts16th)} ({np.mean(maxs_forecasts16th>max_era5_2026)*100:.2f}%) (max={np.max(maxs_forecasts16th)-273.15:.2f}$^\circ$C)")
print(f"Amount of members reaching ERA5's May max d'ERA5 ({max_era5_2026-273.15:.2f}$^\circ$C) for a forecast started on (17/05/2026) : {np.sum(maxs_forecasts17th>max_era5_2026)}/{len(maxs_forecasts17th)} ({np.mean(maxs_forecasts17th>max_era5_2026)*100:.2f}%) (max={np.max(maxs_forecasts17th)-273.15:.2f}$^\circ$C)")
print(f"Amount of members reaching ERA5's May max d'ERA5 ({max_era5_2026-273.15:.2f}$^\circ$C) for a forecast started on (18/05/2026) : {np.sum(maxs_forecasts18th>max_era5_2026)}/{len(maxs_forecasts18th)} ({np.mean(maxs_forecasts18th>max_era5_2026)*100:.2f}%) (max={np.max(maxs_forecasts18th)-273.15:.2f}$^\circ$C)")
print(f"Amount of members reaching ERA5's May max d'ERA5 ({max_era5_2026-273.15:.2f}$^\circ$C) for a forecast started on (19/05/2026) : {np.sum(maxs_forecasts19th>max_era5_2026)}/{len(maxs_forecasts19th)} ({np.mean(maxs_forecasts19th>max_era5_2026)*100:.2f}%) (max={np.max(maxs_forecasts19th)-273.15:.2f}$^\circ$C)")
print(f"Amount of members reaching ERA5's May max d'ERA5 ({max_era5_2026-273.15:.2f}$^\circ$C) for a forecast started on (20/05/2026) : {np.sum(maxs_forecasts20th>max_era5_2026)}/{len(maxs_forecasts20th)} ({np.mean(maxs_forecasts20th>max_era5_2026)*100:.2f}%) (max={np.max(maxs_forecasts20th)-273.15:.2f}$^\circ$C)")
print(f"Amount of members reaching ERA5's May max d'ERA5 ({max_era5_2026-273.15:.2f}$^\circ$C) for a forecast started on (21/05/2026) : {np.sum(maxs_forecasts21th>max_era5_2026)}/{len(maxs_forecasts21th)} ({np.mean(maxs_forecasts21th>max_era5_2026)*100:.2f}%) (max={np.max(maxs_forecasts21th)-273.15:.2f}$^\circ$C)")
print(f"Amount of members reaching ERA5's May max d'ERA5 ({max_era5_2026-273.15:.2f}$^\circ$C) for a forecast started on (22/05/2026) : {np.sum(maxs_forecasts22th>max_era5_2026)}/{len(maxs_forecasts22th)} ({np.mean(maxs_forecasts22th>max_era5_2026)*100:.2f}%) (max={np.max(maxs_forecasts22th)-273.15:.2f}$^\circ$C)")
print(f"Amount of members reaching ERA5's May max d'ERA5 ({max_era5_2026-273.15:.2f}$^\circ$C) for a forecast started on (23/05/2026) : {np.sum(maxs_forecasts23th>max_era5_2026)}/{len(maxs_forecasts23th)} ({np.mean(maxs_forecasts23th>max_era5_2026)*100:.2f}%) (max={np.max(maxs_forecasts23th)-273.15:.2f}$^\circ$C)")

In [ ]:
temp=datasets_results['2026051700']['2t'].isel(values=[paris_index]).isel(step=range(0,76)).mean("values").max("step").compute()-273.15
temp.sortby(temp,ascending=False)

In [ ]:
MSE_compute_and_plot(datasets_results["2026051700"].sel(number=171),title="MSE_member_171_init_17th",region=[paris_index])

## Plotting wind

In [ ]:
def generate_wind_map(target_init, target_member, target_valid_date, levels, datasets_results):
    """Generate an interactive HTML wind map using Leaflet and leaflet-velocity.

    Extracts wind components (u, v) for specified pressure levels on a target valid date, 
    interpolates them to a regular grid, and exports the data into an interactive HTML map.

    Args:
        target_init: String representing the forecast initialization date.
        target_member: Integer representing the specific ensemble member to plot.
        target_valid_date: pandas Timestamp of the valid date to plot.
        levels: List of pressure levels (in hPa) to extract wind data for.
        datasets_results: Dictionary containing the xarray Datasets keyed by initialization date.

    Returns:
        None. Saves an HTML file containing the interactive wind map.
    """
    print(f"Init: {target_init}\nMember: {target_member}")

    #EXTRACT DATA
    ds = datasets_results[target_init]
    LAT = ds["latitude"].values
    LON = ds["longitude"].values

    lon_min, lon_max = LON.min(), LON.max()
    lat_min, lat_max = LAT.min(), LAT.max()

    valid_dates = pd.to_datetime(ds.valid_time.values).floor("D")
    idx = np.where(valid_dates == target_valid_date)[0]

    grid_res = 0.5
    all_wind_data = {}

    #PREPARE DATA FOR LEAFLET VELOCITY
    for level in levels:
        u_raw = ds[f"u_{level}"].sel(number=target_member).isel(step=idx).mean("step").values
        v_raw = ds[f"v_{level}"].sel(number=target_member).isel(step=idx).mean("step").values

        GRID_LON, GRID_LAT, U_2D = interpolate_to_grid(LON, LAT, u_raw, interp_type='linear', hres=grid_res)
        _, _, V_2D = interpolate_to_grid(LON, LAT, v_raw, interp_type='linear', hres=grid_res)

        grid_lon = GRID_LON[0, :]
        grid_lat = GRID_LAT[:, 0]

        U_2D = np.nan_to_num(U_2D, nan=0.0)
        V_2D = np.nan_to_num(V_2D, nan=0.0)

        wind_data = [
            {
                "header": {
                    "parameterCategory": 2, "parameterNumber": 2,
                    "nx": len(grid_lon), "ny": len(grid_lat),
                    "lo1": float(grid_lon.min()), "lo2": float(grid_lon.max()),
                    "la1": float(grid_lat.max()), "la2": float(grid_lat.min()),
                    "dx": grid_res, "dy": grid_res,
                },
                "data": U_2D[::-1, :].flatten().tolist() 
            },
            {
                "header": {
                    "parameterCategory": 2, "parameterNumber": 3,
                    "nx": len(grid_lon), "ny": len(grid_lat),
                    "lo1": float(grid_lon.min()), "lo2": float(grid_lon.max()),
                    "la1": float(grid_lat.max()), "la2": float(grid_lat.min()),
                    "dx": grid_res, "dy": grid_res,
                },
                "data": V_2D[::-1, :].flatten().tolist()
            }
        ]
        all_wind_data[str(level)] = wind_data

    json_data = json.dumps(all_wind_data)

    html_template = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Winds | {target_valid_date.date()}</title>
    <link rel="stylesheet" href="https://unpkg.com/leaflet@1.7.1/dist/leaflet.css" />
    <script src="https://unpkg.com/leaflet@1.7.1/dist/leaflet.js"></script>
    <link rel="stylesheet" href="https://unpkg.com/leaflet-velocity@1.6.0/dist/leaflet-velocity.css" />
    <script src="https://unpkg.com/leaflet-velocity@1.6.0/dist/leaflet-velocity.js"></script>
    <style>
        body, html, #map {{ width: 100%; height: 100%; margin: 0; padding: 0; background: #111; }}
        
        .leaflet-control-velocity, .legend-box {{
            background: rgba(255, 255, 255, 0.9) !important;
            color: #111 !important;
            box-shadow: 0 0 15px rgba(0,0,0,0.4);
            border: 1px solid #ccc;
            border-radius: 4px;
            font-family: monospace;
            padding: 8px 12px;
        }}
        
        .legend-gradient {{
            width: 250px;
            height: 12px;
            background: linear-gradient(to right, #0d0887, #260394, #3a049a, #4d059e, #6009a0, #720ea0, #83159e, #941d9a, #a52695, #b5308c, #c43a82, #d14677, #de536c, #e86161, #f16f56, #f77f49, #fc903d, #ffa231, #ffb525, #ffc91b, #ffde11, #fbf305, #eef820, #d9ff36, #bfff54, #9eff74, #7efd96, #62f9b8);
            border-radius: 2px;
            margin: 6px 0;
        }}
        
        .legend-labels {{
            display: flex;
            justify-content: space-between;
            font-size: 11px;
            font-weight: bold;
        }}
        
        .leaflet-control-layers {{
            background: rgba(255, 255, 255, 0.95);
            border-radius: 4px;
            font-family: monospace;
            box-shadow: 0 0 15px rgba(0,0,0,0.4);
        }}
    </style>
</head>
<body>
    <div id="map"></div>
    <script>
        var allWindData = {json_data};
        
        var map = L.map('map', {{ 
            zoomControl: false,
            minZoom: 3,
            maxZoom: 7,
            zoomSnap: 0,
            zoomDelta: 0.5,
            wheelPxPerZoomLevel: 60
        }}).setView([{(lat_min+lat_max)/2}, {(lon_min+lon_max)/2}], 4);
        
        L.tileLayer('https://{{s}}.basemaps.cartocdn.com/light_all/{{z}}/{{x}}/{{y}}{{r}}.png', {{
            attribution: '&copy; OpenStreetMap &copy; CARTO'
        }}).addTo(map);
        
        var colorScaleArray = [
            "#0d0887", "#260394", "#3a049a", "#4d059e", "#6009a0", "#720ea0", "#83159e", "#941d9a", 
            "#a52695", "#b5308c", "#c43a82", "#d14677", "#de536c", "#e86161", "#f16f56", "#f77f49", 
            "#fc903d", "#ffa231", "#ffb525", "#ffc91b", "#ffde11", "#fbf305", "#eef820", "#d9ff36", 
            "#bfff54", "#9eff74", "#7efd96", "#62f9b8"
        ];

        var layerControlGroups = {{}};
        var firstLayerAdded = false;

        for (var level in allWindData) {{
            var vLayer = L.velocityLayer({{
                displayValues: true,
                displayOptions: {{
                    velocityType: level + 'hPa Wind',
                    displayPosition: 'bottomleft',
                    displayEmptyString: 'No wind data'
                }},
                data: allWindData[level],
                maxVelocity: 55,
                velocityScale: 0.01,
                particleMultiplier: 1 / 120,
                particleAge: 120,
                lineWidth: 1.5,
                colorScale: colorScaleArray
            }});
            
            layerControlGroups[level + " hPa"] = vLayer;
            
            if (!firstLayerAdded) {{
                vLayer.addTo(map);
                firstLayerAdded = true;
            }}
        }}

        L.control.layers(layerControlGroups, null, {{position: 'topright', collapsed: false}}).addTo(map);

        var legend = L.control({{position: 'bottomright'}});
        legend.onAdd = function (map) {{
            var div = L.DomUtil.create('div', 'legend-box');
            div.innerHTML = '<strong>Wind Speed (m/s)</strong>' +
                            '<div class="legend-gradient"></div>' +
                            '<div class="legend-labels"><span>0</span><span>55+</span></div>';
            return div;
        }};
        legend.addTo(map);

    </script>
</body>
</html>
"""

    with open(f"wind_htmls/wind_map_target_{target_valid_date.date()}_init_{target_init}_member_{target_member}_multilevel.html", "w") as f:
        f.write(html_template)

In [ ]:
m = int(members_with_low_pressure_area_top10["2026051800"][0])
target_valid_date = pd.Timestamp("2026-05-29")
generate_wind_map("2026051800", m, target_valid_date, [1000,925,850,700,600,500,400,300,250,200], datasets_results)
target_valid_date = pd.Timestamp("2026-05-28")
generate_wind_map("2026051800", m, target_valid_date, [1000,925,850,700,600,500,400,300,250,200], datasets_results)
target_valid_date = pd.Timestamp("2026-05-27")
generate_wind_map("2026051800", m, target_valid_date, [1000,925,850,700,600,500,400,300,250,200], datasets_results)
target_valid_date = pd.Timestamp("2026-05-26")
generate_wind_map("2026051800", m, target_valid_date, [1000,925,850,700,600,500,400,300,250,200], datasets_results)

In [ ]:
m = np.random.choice(members_without_low_pressure_area["2026051800"])
target_valid_date = pd.Timestamp("2026-05-29")
generate_wind_map("2026051800", m, target_valid_date, [1000,925,850,700,600,500,400,300,250,200], datasets_results)
target_valid_date = pd.Timestamp("2026-05-28")
generate_wind_map("2026051800", m, target_valid_date, [1000,925,850,700,600,500,400,300,250,200], datasets_results)
target_valid_date = pd.Timestamp("2026-05-27")
generate_wind_map("2026051800", m, target_valid_date, [1000,925,850,700,600,500,400,300,250,200], datasets_results)
target_valid_date = pd.Timestamp("2026-05-26")
generate_wind_map("2026051800", m, target_valid_date, [1000,925,850,700,600,500,400,300,250,200], datasets_results)

## Quantifying role of cutoff low

In [ ]:
region=plot_region_indexs
N_boot = 2000
N_resample=10

In [ ]:
#DATA
v_17th = datasets_results["2026051700"].sel(number=members_exceeding['2026051700'])
time_mask_17th = (pd.to_datetime(v_17th.valid_time.values).date == pd.to_datetime("2026-05-25").date())

# BOOTSTRAP
t2m_mean_on_target_day_per_member_17th = (
    v_17th["2t"]
    .isel(values=region)
    .isel(step=time_mask_17th)
    .mean(dim="step")
).compute() - 273.15

data_top10_cutoff_17th = t2m_mean_on_target_day_per_member_17th.sel(number=members_with_high_D4_17th_top10) #members_with_low_pressure_area_17th_top10
mean_top10_cutoff_17th = data_top10_cutoff_17th.mean(dim="number")
mean_all_17th = t2m_mean_on_target_day_per_member_17th.mean(dim="number")

count_hotter_17th = xr.zeros_like(mean_top10_cutoff_17th)

N_resample=len(members_with_high_D4_17th_top10)

for i in range(N_boot):
    members_sampled = np.random.choice(t2m_mean_on_target_day_per_member_17th.number.values, size=N_resample, replace=False)
    sample = t2m_mean_on_target_day_per_member_17th.sel(number=members_sampled)
    sample_mean=sample.mean(dim="number")
    count_hotter_17th += (sample_mean >= mean_top10_cutoff_17th)

p_values_17th = count_hotter_17th / N_boot
anomaly_17th = mean_top10_cutoff_17th-mean_all_17th

In [ ]:
#DATA
v_18th = datasets_results["2026051800"].sel(number=members_exceeding['2026051800'])
time_mask_18th = (pd.to_datetime(v_18th.valid_time.values).date == pd.to_datetime("2026-05-25").date())

# BOOTSTRAP
t2m_mean_on_target_day_per_member_18th = (
    v_18th["2t"]
    .isel(values=region)
    .isel(step=time_mask_18th)
    .mean(dim="step")
).compute() - 273.15

data_top10_cutoff_18th = t2m_mean_on_target_day_per_member_18th.sel(number=members_with_high_D4_18th_top10)
mean_top10_cutoff_18th = data_top10_cutoff_18th.mean(dim="number")
mean_all_18th = t2m_mean_on_target_day_per_member_18th.mean(dim="number")

count_hotter_18th = xr.zeros_like(mean_top10_cutoff_18th)

N_resample=len(members_with_high_D4_18th_top10)

for i in range(N_boot):
    members_sampled = np.random.choice(t2m_mean_on_target_day_per_member_18th.number.values, size=N_resample, replace=False)
    sample = t2m_mean_on_target_day_per_member_18th.sel(number=members_sampled)
    sample_mean=sample.mean(dim="number")
    count_hotter_18th += (sample_mean >= mean_top10_cutoff_18th)

p_values_18th = count_hotter_18th / N_boot
anomaly_18th = mean_top10_cutoff_18th-mean_all_18th

In [ ]:
plot_lons = LON[region]
plot_lats = LAT[region]
lon_min, lon_max = np.min(plot_lons), np.max(plot_lons)
lat_min, lat_max = np.min(plot_lats), np.max(plot_lats)

anom_min = min(anomaly_17th.min(), anomaly_18th.min())
anom_max = max(anomaly_17th.max(), anomaly_18th.max())

fig, axes = plt.subplots(
    nrows=2, ncols=2, 
    figsize=get_figsize(WIDTH_INSA, fraction=1, height_factor=0.65), 
    subplot_kw={'projection': ccrs.PlateCarree()}
)

## LEFT PLOT
ax1 = axes[0,0]
ax1.coastlines()
ax1.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
cf_anomaly = ax1.tripcolor(
    plot_lons, plot_lats, anomaly_17th,
    cmap=ncp.pal("Cobalt"),
    transform=ccrs.PlateCarree(),
    vmin=anom_min, vmax=anom_max,
)

mask = p_values_17th <= 0.05
ax1.scatter(
    plot_lons[mask], plot_lats[mask], 
    color="black", s=0.2, alpha=0.6, 
    transform=ccrs.PlateCarree(),
    label="Significantly hotter (p $\leq$ 0.05)",
    linewidths=0,
)
ax1.legend(loc="lower right", frameon=True, facecolor="white", framealpha=0.9)
ax1.set_title("(a)")
fig.colorbar(cf_anomaly, ax=ax1, orientation="vertical", label="Anomaly of top 10 cutoff\nwrt all the members ($^\circ$C)", shrink=0.6)

## RIGHT PLOT

ax2 = axes[0,1]
ax2.coastlines()
ax2.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
p_levels = [0, 0.01, 0.05, 1.0]

cmap_neon_nicopal=mcolors.ListedColormap(["#000286","#c067ec","#ffffff"])
cf_pval = ax2.tricontourf(
    plot_lons, plot_lats, p_values_17th,
    levels=p_levels,
    cmap=cmap_neon_nicopal,
    norm=mcolors.BoundaryNorm(p_levels, cmap_neon_nicopal.N),
    transform=ccrs.PlateCarree(),
)

ax2.set_title("(b)")
fig.colorbar(cf_pval, ax=ax2, orientation="vertical", label="p-value", shrink=0.6)

## LEFT PLOT
ax1 = axes[1,0]
ax1.coastlines()
ax1.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
cf_anomaly = ax1.tripcolor(
    plot_lons, plot_lats, anomaly_18th,
    cmap=ncp.pal("Cobalt"),
    transform=ccrs.PlateCarree(),
    vmin=anom_min, vmax=anom_max,
)

mask = p_values_18th <= 0.05
ax1.scatter(
    plot_lons[mask], plot_lats[mask], 
    color="black", s=0.2, alpha=0.6, 
    transform=ccrs.PlateCarree(),
    label="Significantly hotter (p $\leq$ 0.05)",
    linewidths=0,
)
ax1.legend(loc="lower right", frameon=True, facecolor="white", framealpha=0.9)
ax1.set_title("(c)")
fig.colorbar(cf_anomaly, ax=ax1, orientation="vertical", label="Anomaly of top 10 cutoff activity\nwrt all the members (exceeding threshold) ($^\circ$C)", shrink=0.6)

## RIGHT PLOT

ax2 = axes[1,1]
ax2.coastlines()
ax2.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
p_levels = [0, 0.01, 0.05, 1.0]

cmap_neon_nicopal=mcolors.ListedColormap(["#000286","#c067ec","#ffffff"])
cf_pval = ax2.tricontourf(
    plot_lons, plot_lats, p_values_18th,
    levels=p_levels,
    cmap=cmap_neon_nicopal,
    norm=mcolors.BoundaryNorm(p_levels, cmap_neon_nicopal.N),
    transform=ccrs.PlateCarree(),
)

ax2.set_title("(d)")
fig.colorbar(cf_pval, ax=ax2, orientation="vertical", label="p-value", shrink=0.6)

plt.tight_layout()
plt.savefig("MAY_2026_HEATWAVE/pvalues.pdf", bbox_inches="tight")
plt.show()

In [ ]:
idx = average_region[region]

# 17th
sig_17 = anomaly_17th.isel(values=idx).where(p_values_17th.isel(values=idx) <= 0.05)
print("=== MAY 17th INIT EXACERBATION (CENTER REGION) ===")
print(f"Mean: {sig_17.mean().values:.2f}°C | Median: {sig_17.median().values:.2f}°C")
print(f"Local range: {sig_17.min().values:.2f}°C up to {sig_17.max().values:.2f}°C")
print(f"5th-95th spatial percentile: {sig_17.quantile(0.05).values:.2f}°C to {sig_17.quantile(0.95).values:.2f}°C\n")

# 18th
sig_18 = anomaly_18th.isel(values=idx).where(p_values_18th.isel(values=idx) <= 0.05)
print("=== MAY 18th INIT EXACERBATION (CENTER REGION) ===")
print(f"Mean: {sig_18.mean().values:.2f}°C | Median: {sig_18.median().values:.2f}°C")
print(f"Local range: {sig_18.min().values:.2f}°C up to {sig_18.max().values:.2f}°C")
print(f"5th-95th spatial percentile: {sig_18.quantile(0.05).values:.2f}°C to {sig_18.quantile(0.95).values:.2f}°C\n")


## PDF shift

In [ ]:
# 17th data
t_17 = (datasets_results["2026051700"]["2t"].isel(values=average_region).mean("values").swap_dims({"step": "valid_time"}).sel(valid_time="2026-05-25").mean("valid_time").compute()) - 273.15
z_17 = (datasets_results["2026051700"]["z_500"].isel(values=average_region).mean("values").swap_dims({"step": "valid_time"}).sel(valid_time="2026-05-25").mean("valid_time").compute()) / 9.81

# 18th data
t_18 = (datasets_results["2026051800"]["2t"].isel(values=average_region).mean("values").swap_dims({"step": "valid_time"}).sel(valid_time="2026-05-25").mean("valid_time").compute()) - 273.15
z_18 = (datasets_results["2026051800"]["z_500"].isel(values=average_region).mean("values").swap_dims({"step": "valid_time"}).sel(valid_time="2026-05-25").mean("valid_time").compute()) / 9.81

# WITH Z500 Threshold (Exceeding Only)
t_17_with_exc = t_17.sel(number=members_with_low_pressure_area_17th).values
t_17_without_exc = t_17.sel(number=members_without_low_pressure_area_17th).values
z_17_with_exc = z_17.sel(number=members_with_low_pressure_area_17th).values
z_17_without_exc = z_17.sel(number=members_without_low_pressure_area_17th).values

t_18_with_exc = t_18.sel(number=members_with_low_pressure_area_18th).values
t_18_without_exc = t_18.sel(number=members_without_low_pressure_area_18th).values
z_18_with_exc = z_18.sel(number=members_with_low_pressure_area_18th).values
z_18_without_exc = z_18.sel(number=members_without_low_pressure_area_18th).values


# WITHOUT Z500 Threshold (All members)
t_17_with_all = t_17.sel(number=members_with_low_pressure_area_17th_all).values
t_17_without_all = t_17.sel(number=members_without_low_pressure_area_17th_all).values
z_17_with_all = z_17.sel(number=members_with_low_pressure_area_17th_all).values
z_17_without_all = z_17.sel(number=members_without_low_pressure_area_17th_all).values

t_18_with_all = t_18.sel(number=members_with_low_pressure_area_18th_all).values
t_18_without_all = t_18.sel(number=members_without_low_pressure_area_18th_all).values
z_18_with_all = z_18.sel(number=members_with_low_pressure_area_18th_all).values
z_18_without_all = z_18.sel(number=members_without_low_pressure_area_18th_all).values

# export data for maeve

cols

In [ ]:
# already done above

values over the target region

In [ ]:
out_dir = "data_maeve"
os.makedirs(out_dir, exist_ok=True)

inits = ["2026051700", "2026051800"]

for init in inits:
    print(f"Processing init: {init}...")

    t2m_mean_daily = (
        datasets_results[init]["2t"]
        .isel(values=average_region)
        .mean("values")
        .swap_dims({"step": "valid_time"})
        .sel(valid_time="2026-05-25")
        .mean("valid_time")
        .compute()
    ) - 273.15

    t2m_mean_12z = (
        datasets_results[init]["2t"]
        .isel(values=average_region)
        .mean("values")
        .swap_dims({"step": "valid_time"})
        .sel(valid_time="2026-05-25T12:00")
        .compute()
    ) - 273.15

    t2m_max_daily = (
        datasets_results[init]["2t"]
        .isel(values=average_region)
        .swap_dims({"step": "valid_time"})
        .sel(valid_time="2026-05-25")
        .mean("valid_time")
        .max("values")
        .compute()
    ) - 273.15

    t2m_max_12z = (
        datasets_results[init]["2t"]
        .isel(values=average_region)
        .swap_dims({"step": "valid_time"})
        .sel(valid_time="2026-05-25T12:00")
        .max("values")
        .compute()
    ) - 273.15

    z500_mean_daily = (
        datasets_results[init]["z_500"]
        .isel(values=average_region)
        .mean("values")
        .swap_dims({"step": "valid_time"})
        .sel(valid_time="2026-05-25")
        .mean("valid_time")
        .compute()
    ) / 9.81

    df_export = pd.DataFrame({
        "Init_Date": init,
        "Member": t2m_mean_daily.number.values,
        "T2m_Regional_Mean_Daily_C": t2m_mean_daily.values,
        "T2m_Regional_Mean_12UTC_C": t2m_mean_12z.values,
        "T2m_Gridcell_Max_Daily_C": t2m_max_daily.values,
        "T2m_Gridcell_Max_12UTC_C": t2m_max_12z.values,
        "Z500_Regional_Mean_Daily_gpm": z500_mean_daily.values
    })

    out_file = f"{out_dir}/regional_metrics_Init_{init}_20260525.csv"
    df_export.to_csv(out_file, index=False)

significance

In [ ]:
v_17 = datasets_results["2026051700"]
time_mask_17 = (
    pd.to_datetime(v_17.valid_time.values).date == pd.to_datetime("2026-05-25").date()
)
z_17_top10 = (
    v_17["z_500"]
    .isel(values=region)
    .isel(step=time_mask_17)
    .sel(number=members_with_high_D4_17th_top10)
    .mean(["step", "number"])
    / 9.81
).compute()

v_18 = datasets_results["2026051800"]
time_mask_18 = (
    pd.to_datetime(v_18.valid_time.values).date == pd.to_datetime("2026-05-25").date()
)
z_18_top10 = (
    v_18["z_500"]
    .isel(values=region)
    .isel(step=time_mask_18)
    .sel(number=members_with_high_D4_18th_top10)
    .mean(["step", "number"])
    / 9.81
).compute()

ds_sig = xr.merge(
    [
        p_values_17th.rename("p_values_17th"),
        anomaly_17th.rename("anomaly_17th_C"),
        z_17_top10.rename("z500_top10_17th"),
        p_values_18th.rename("p_values_18th"),
        anomaly_18th.rename("anomaly_18th_C"),
        z_18_top10.rename("z500_top10_18th"),
    ],
    compat="override",
)

ds_sig = ds_sig.assign_coords(
    latitude=(["values"], LAT[region]), longitude=(["values"], LON[region])
)

ds_sig.to_netcdf(f"{out_dir}/significance_cutoff_vs_no_cutoff.nc")

vertical temp profile

In [ ]:
out_dir = "data_maeve"
os.makedirs(out_dir, exist_ok=True)

levels = [1000, 925, 850, 700, 600, 500, 400, 300, 250, 200]
t_vars = [f"t_{lvl}" for lvl in levels]
inits = ["2026051700", "2026051800"]
N_boot = 2000

for init in inits:
    print(f"Processing vertical profiles & significance for init: {init}...")
    v = datasets_results[init]

    level_data = []
    for lvl, var in zip(levels, t_vars):
        t_mean = v[var].isel(values=average_region).mean(dim="values")
        level_data.append(t_mean.expand_dims(level=[lvl]))

    da_init = xr.concat(level_data, dim="level")
    da_init = da_init - 273.15

    if "step" in da_init.dims:
        da_init = da_init.swap_dims({"step": "valid_time"})

    exc_members = members_exceeding[init]
    top10_members = (
        members_with_high_D4_17th_top10
        if init == "2026051700"
        else members_with_high_D4_18th_top10
    )

    da_exc = da_init.sel(number=exc_members)
    mean_top10 = da_init.sel(number=top10_members).mean(dim="number")

    count_hotter = xr.zeros_like(mean_top10)
    N_resample = len(top10_members)

    for i in range(N_boot):
        members_sampled = np.random.choice(exc_members, size=N_resample, replace=False)
        sample_mean = da_exc.sel(number=members_sampled).mean(dim="number")
        count_hotter += sample_mean >= mean_top10

    p_values = count_hotter / N_boot

    ds_out = xr.Dataset({"temperature": da_init, "p_values": p_values})

    ds_out.attrs["description"] = (
        f"Vertical temperature profiles & significance averaged over target region. Init: {init}."
    )
    ds_out.attrs["units"] = "Celsius"

    out_path = f"{out_dir}/vertical_temperature_profiles_Init_{init}.nc"
    ds_out.to_netcdf(out_path)

pdf shift

In [ ]:
out_dir = "data_maeve"

df_export = pd.DataFrame({
    "T2m_With_Cutoff": pd.Series(np.concatenate([t_17_with_exc, t_18_with_exc])),
    "T2m_Without_Cutoff": pd.Series(np.concatenate([t_17_without_exc, t_18_without_exc])),
    "Z500_With_Cutoff": pd.Series(np.concatenate([z_17_with_exc, z_18_with_exc])),
    "Z500_Without_Cutoff": pd.Series(np.concatenate([z_17_without_exc, z_18_without_exc]))
})

out_file = f"{out_dir}/pdf_data_with_threshold.csv"
df_export.to_csv(out_file, index=False)

return times

In [ ]:
df_boosted = pd.DataFrame({
    "Return_Period_Years": rp,
    "Temperature_C": T_arr,
    "CI_Lower_C": low,
    "CI_Upper_C": up
})
df_boosted.to_csv(f"{out_dir}/return_times_boosted_estimator.csv", index=False)

obs_temp = np.sort(GEV.extremes.values)
obs_rp = (len(obs_temp) + 1.0) / np.arange(len(obs_temp), 0, -1)
df_hist = pd.DataFrame({
    "Return_Period_Years": obs_rp, 
    "Temperature_C": obs_temp
})
df_hist.to_csv(f"{out_dir}/return_times_historical_era5.csv", index=False)

gev_rp_array = np.logspace(0, 4.7, 100) 
gev_temp_array, gev_lower, gev_upper = GEV.get_return_value(gev_rp_array, alpha=0.95)
df_gev = pd.DataFrame({
    "Return_Period_Years": gev_rp_array,
    "Temperature_C": gev_temp_array,
    "CI_Lower_C": gev_lower,
    "CI_Upper_C": gev_upper
})
df_gev.to_csv(f"{out_dir}/return_times_gev_fit.csv", index=False)

# Figure

In [ ]:
threshold_t2m = 19

t_with_all = np.concatenate([t_17_with_all, t_18_with_all])
t_without_all = np.concatenate([t_17_without_all, t_18_without_all])

t_with_exc = np.concatenate([t_17_with_exc, t_18_with_exc])
t_without_exc = np.concatenate([t_17_without_exc, t_18_without_exc])

print("=== ALL MEMBERS ===")
print(f"Probability of T2m >= {threshold_t2m}°C on May 25th:")
print(f"With cutoff:    {np.mean(t_with_all >= threshold_t2m) * 100:.1f}% ({np.sum(t_with_all >= threshold_t2m)}/{len(t_with_all)})")
print(f"Without cutoff: {np.mean(t_without_all >= threshold_t2m) * 100:.1f}% ({np.sum(t_without_all >= threshold_t2m)}/{len(t_without_all)})\n")

print("=== EXCEEDING Z500 THRESHOLD ONLY ===")
print(f"Probability of T2m >= {threshold_t2m}°C on May 25th:")
print(f"With cutoff:    {np.mean(t_with_exc >= threshold_t2m) * 100:.1f}% ({np.sum(t_with_exc >= threshold_t2m)}/{len(t_with_exc)})")
print(f"Without cutoff: {np.mean(t_without_exc >= threshold_t2m) * 100:.1f}% ({np.sum(t_without_exc >= threshold_t2m)}/{len(t_without_exc)})")

In [ ]:
out_dir = "data_maeve"

setup_latex_style(8)

# load exported data
ds_sig = xr.open_dataset(f"{out_dir}/significance_cutoff_vs_no_cutoff.nc")
ds_v17 = xr.open_dataset(f"{out_dir}/vertical_temperature_profiles_Init_2026051700.nc")
ds_v18 = xr.open_dataset(f"{out_dir}/vertical_temperature_profiles_Init_2026051800.nc")
df_pdf = pd.read_csv(f"{out_dir}/pdf_data_with_threshold.csv")

# recreate all exceeding members lists using the D4 variables
exc_17 = np.concatenate([members_with_high_D4_17th, members_without_high_D4_17th])
exc_18 = np.concatenate([members_with_high_D4_18th, members_without_high_D4_18th])

# calc hovmoller diffs (top 10 activity - all exceeding members)
prof_17_diff = ds_v17.temperature.sel(
    number=members_with_high_D4_17th_top10
).mean("number") - ds_v17.temperature.sel(
    number=exc_17
).mean("number")

prof_18_diff = ds_v18.temperature.sel(
    number=members_with_high_D4_18th_top10
).mean("number") - ds_v18.temperature.sel(
    number=exc_18
).mean("number")

prof_17_diff = prof_17_diff.sel(valid_time=slice(None, "2026-06-01"))
prof_18_diff = prof_18_diff.sel(valid_time=slice(None, "2026-06-01"))

# setup layout (increased height and hspace to accommodate dates on both rows)
fig = plt.figure(figsize=get_figsize(WIDTH_INSA, fraction=1, height_factor=0.55))
gs = fig.add_gridspec(2, 3, width_ratios=[1.4, 1.2, 1.2], hspace=0.65, wspace=0.35)

# config for maps
lons, lats = ds_sig.longitude.values, ds_sig.latitude.values
lon_min, lon_max = np.min(lons), np.max(lons)
lat_min, lat_max = np.min(lats), np.max(lats)

anom_max = float(max(np.abs(ds_sig.anomaly_17th_C).max(), np.abs(ds_sig.anomaly_18th_C).max()))

hov_vmax = float(max(np.abs(prof_17_diff).max(), np.abs(prof_18_diff).max()))
hov_norm = mcolors.TwoSlopeNorm(vmin=-hov_vmax, vcenter=0, vmax=hov_vmax)
hov_levels = np.linspace(-hov_vmax, hov_vmax, 21)

# ---------------------------------------------------------
# 17th INIT ROW
# ---------------------------------------------------------

ax1 = fig.add_subplot(gs[0, 0], projection=ccrs.PlateCarree())
ax1.coastlines()
ax1.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
cf1 = ax1.tripcolor(
    lons, lats, ds_sig.anomaly_17th_C, cmap=ncp.pal("Cobalt"),
    vmin=-anom_max, vmax=anom_max, transform=ccrs.PlateCarree()
)

contours_z17 = ax1.tricontour(
    lons, lats, ds_sig.z500_top10_17th,
    levels=12, colors="black", linewidths=0.6, transform=ccrs.PlateCarree()
)
ax1.clabel(contours_z17, fmt="%.0f", inline=True, fontsize=7)

mask17 = ds_sig.p_values_17th <= 0.05
ax1.scatter(
    lons[~mask17], lats[~mask17], color="black", s=0.2, alpha=0.6,
    label="$p>0.05$", transform=ccrs.PlateCarree(), linewidths=0
)
ax1.legend(loc="lower right", frameon=True, facecolor="white", framealpha=0.9, prop={"size": 6})
ax1.set_title("(a)")

ax2 = fig.add_subplot(gs[0, 1])
cf2 = ax2.contourf(
    prof_17_diff.valid_time, prof_17_diff.level, prof_17_diff,
    cmap=ncp.pal("Cobalt"), levels=hov_levels, norm=hov_norm
)

mask17_prof = ds_v17.p_values.sel(valid_time=slice(None, "2026-06-01")).values <= 0.05
T_mesh17, P_mesh17 = np.meshgrid(prof_17_diff.valid_time.values, prof_17_diff.level.values)
ax2.scatter(
    T_mesh17[~mask17_prof], P_mesh17[~mask17_prof], color="black", s=0.2, alpha=0.6,
    linewidths=0, label="$p>0.05$"
)

ax2.invert_yaxis()
ax2.set_ylabel("Pressure (hPa)")
ax2.set_title("(b)")
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%d-%m"))
ax2.legend(loc="lower right", frameon=True, facecolor="white", framealpha=0.9, prop={"size": 6})

# ---------------------------------------------------------
# 18th INIT ROW
# ---------------------------------------------------------

ax3 = fig.add_subplot(gs[1, 0], projection=ccrs.PlateCarree())
ax3.coastlines()
ax3.set_extent([lon_min, lon_max, lat_min, lat_max], crs=ccrs.PlateCarree())
cf3 = ax3.tripcolor(
    lons, lats, ds_sig.anomaly_18th_C, cmap=ncp.pal("Cobalt"),
    vmin=-anom_max, vmax=anom_max, transform=ccrs.PlateCarree()
)

contours_z18 = ax3.tricontour(
    lons, lats, ds_sig.z500_top10_18th,
    levels=12, colors="black", linewidths=0.6, transform=ccrs.PlateCarree()
)
ax3.clabel(contours_z18, fmt="%.0f", inline=True, fontsize=7)

mask18 = ds_sig.p_values_18th <= 0.05
ax3.scatter(
    lons[~mask18], lats[~mask18], color="black", s=0.2, alpha=0.6,
    label="$p>0.05$", transform=ccrs.PlateCarree(), linewidths=0
)
ax3.legend(loc="lower right", frameon=True, facecolor="white", framealpha=0.9, prop={"size": 6})
ax3.set_title("(c)")

ax4 = fig.add_subplot(gs[1, 1])
cf4 = ax4.contourf(
    prof_18_diff.valid_time, prof_18_diff.level, prof_18_diff,
    cmap=ncp.pal("Cobalt"), levels=hov_levels, norm=hov_norm
)

mask18_prof = ds_v18.p_values.sel(valid_time=slice(None, "2026-06-01")).values <= 0.05
T_mesh18, P_mesh18 = np.meshgrid(prof_18_diff.valid_time.values, prof_18_diff.level.values)
ax4.scatter(
    T_mesh18[~mask18_prof], P_mesh18[~mask18_prof], color="black", s=0.2, alpha=0.6,
    linewidths=0, label="$p>0.05$"
)

ax4.invert_yaxis()
ax4.set_ylabel("Pressure (hPa)")
ax4.set_title("(d)")
ax4.xaxis.set_major_formatter(mdates.DateFormatter("%d-%m"))
ax4.legend(loc="lower right", frameon=True, facecolor="white", framealpha=0.9, prop={"size": 6})

for ax in [ax2, ax4]:
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

# Shared colorbars anchored to the bottom row
cb1 = fig.colorbar(
    cf3, ax=[ax1, ax3], shrink=0.85, label="Anomaly ($^\\circ$C)",
    orientation="horizontal", pad=0.12, ticks=mtickers.MaxNLocator(nbins=7)
)
cb2 = fig.colorbar(
    cf4, ax=[ax2, ax4], shrink=0.85, label="$\\Delta$T ($^\\circ$C)",
    orientation="horizontal", pad=0.12, ticks=mtickers.MaxNLocator(nbins=5)
)

# ---------------------------------------------------------
# PDF SHIFT 
# ---------------------------------------------------------

T_with = df_pdf["T2m_With_Cutoff"].dropna().values
T_without = df_pdf["T2m_Without_Cutoff"].dropna().values
T = np.concatenate([T_with, T_without])

has_cutoff = np.concatenate([np.ones(len(T_with), dtype=bool), np.zeros(len(T_without), dtype=bool)])
Ta = T - T.mean()

def summ(x, g1, g2):
    if g1.sum() < 2 or g2.sum() < 2:
        return np.nan, np.nan, np.nan
    d = x[g1].mean() - x[g2].mean()
    wp = stats.ttest_ind(x[g1], x[g2], equal_var=False).pvalue
    ks = stats.ks_2samp(x[g1], x[g2]).pvalue
    return d, wp, ks

def fit_skewnorm(v):
    v = np.asarray(v, float)
    if len(v) < 3:
        return 1, np.mean(v), 1
    mu, s = v.mean(), v.std()
    raw = np.mean((v - mu) ** 3)
    sk = np.clip(abs(raw / max(s**3, 1e-6)), 1e-3, 0.99) * np.sign(raw)
    delta = np.sqrt(
        np.pi / 2 * abs(sk) ** (2 / 3) / (abs(sk) ** (2 / 3) + ((4 - np.pi) / 2) ** (2 / 3))
    ) * np.sign(sk)
    a = delta / np.sqrt(1 - delta**2)
    scale = s / np.sqrt(max(1 - 2 * delta**2 / np.pi, 1e-6))
    return a, mu - scale * delta * np.sqrt(2 / np.pi), scale

ax5 = fig.add_subplot(gs[0, 2])
xs = np.linspace(Ta.min() - 1, Ta.max() + 1, 300)
groups = [(has_cutoff, "#d62728", "cut-off"), (~has_cutoff, "#1f77b4", "no cut-off")]

for g, c, nm in groups:
    if g.sum() > 0:
        ax5.hist(Ta[g], bins=10, density=True, alpha=0.35, color=c)
        sh, lo, sc = fit_skewnorm(Ta[g])
        ax5.plot(
            xs, stats.skewnorm.pdf(xs, sh, loc=lo, scale=sc), color=c, lw=2, label=f"{nm} (n={g.sum()})"
        )
        ax5.axvline(Ta[g].mean(), color=c, ls="--", lw=1)

d, wp, ks = summ(Ta, has_cutoff, ~has_cutoff)
ax5.set_xlabel("RAW T2m anomaly [K]")
ax5.set_ylabel("Density")
ax5.set_title(f"(e) $\\Delta$={d:+.2f} K | Welch p={wp:.3f}\nKS p={ks:.3f}")
ax5.grid(alpha=0.3)
ax5.legend(loc="lower center")

ax6 = fig.add_subplot(gs[1, 2])
xs = np.linspace(Ta.min() - 1, Ta.max() + 1, 300)
groups = [(has_cutoff, "#d62728", "cut-off"), (~has_cutoff, "#1f77b4", "no cut-off")]

for g, c, nm in groups:
    if g.sum() > 0:
        ax6.hist(Ta[g], bins=10, density=True, alpha=0.35, color=c)
        sh, lo, sc = fit_skewnorm(Ta[g])
        ax6.plot(
            xs, stats.skewnorm.pdf(xs, sh, loc=lo, scale=sc), color=c, lw=2, label=f"{nm} (n={g.sum()})"
        )
        ax6.axvline(Ta[g].mean(), color=c, ls="--", lw=1)

d, wp, ks = summ(Ta, has_cutoff, ~has_cutoff)
ax6.set_xlabel("RAW T2m anomaly [K]")
ax6.set_ylabel("Density")
ax6.set_title(f"(f) $\\Delta$={d:+.2f} K | Welch p={wp:.3f}\nKS p={ks:.3f}")
ax6.grid(alpha=0.3)
ax6.legend(loc="lower center")

plt.savefig("MAY_2026_HEATWAVE/summary_exported_data.pdf", bbox_inches="tight")
plt.show()

In [ ]:
out_dir = "data_maeve"

df_boosted = pd.read_csv(f"{out_dir}/return_times_boosted_estimator.csv")
df_hist = pd.read_csv(f"{out_dir}/return_times_historical_era5.csv")
df_gev = pd.read_csv(f"{out_dir}/return_times_gev_fit.csv")

hline = 31.725372

fig, ax = plt.subplots(figsize=get_figsize(WIDTH_INSA, 1, 0.6))

ax.scatter(
    df_hist["Return_Period_Years"], 
    df_hist["Temperature_C"], 
    color="black", marker="o", label="Historical Data (ERA5)", alpha=0.75
)

ax.plot(
    df_gev["Return_Period_Years"], 
    df_gev["Temperature_C"], 
    color="#F85C50", label="GEV Fit on historical data"
)
ax.fill_between(
    df_gev["Return_Period_Years"], 
    df_gev["CI_Lower_C"], 
    df_gev["CI_Upper_C"], 
    color="#5199FF", alpha=0.25, label="95% CI (GEV)"
)

ax.plot(
    df_boosted["Return_Period_Years"], 
    df_boosted["Temperature_C"], 
    color="black", label="Boosted Estimator using (Bloin-Wibe et al, 2025) + AIFS-Ens"
)
ax.fill_betweenx(
    df_boosted["Temperature_C"], 
    df_boosted["CI_Lower_C"], 
    df_boosted["CI_Upper_C"], 
    color="green", alpha=0.3, label="95% bootstrap CI (AIFS Boosted)"
)

ax.axhline(y=hline, ls="--", label="ERA5 May 12UTC Max (2026)")

ax.set_xscale("log")
ax.set_xlim(1, 1e4)
ax.set_ylim(16, 40)
ax.set_xlabel("Estimated return period (Years)")
ax.set_ylabel("max May 12 UTC temperature ($^\\circ$C)")
ax.grid(True, which="major", ls="-", alpha=0.6)
ax.grid(True, which="minor", ls=":", alpha=0.4)
ax.set_yticks(np.arange(16, 42, 2))
ax.set_yticks(np.arange(16, 41, 1), minor=True)
ax.legend(loc="lower right")

plt.tight_layout()
plt.savefig("MAY_2026_HEATWAVE/return_times_replot_from_csv.pdf", bbox_inches="tight")
plt.show()